In [ ]:
# -*- coding: utf-8 -*-
"""
[Stage 4 v2] Performance Evaluation and Hallucination Audit
=============================================================================
Revisi dari Stage 4 v1 — diperbaiki agar konsisten dengan Stage 2 (DPO
dataset generator) dalam hal scoring, tokenisasi lexical, dan kriteria
deteksi halusinasi.

=============================================================================
RINGKASAN PERBEDAAN vs STAGE 4 v1
=============================================================================

[FIX-WEIGHT-1] BOBOT DAHL DISESUAIKAN dengan bobot S_final Stage 2:
               Stage 4 v1  : W1=0.80 (relevance), W2=0.10, W3=0.10
               Stage 4 v2  : W1=0.40 (relevance), W2=0.40 (grounding),
                              W3=0.20 (lexical)
               ALASAN: Di v1, bobot relevance (0.80) mendominasi sehingga
               respons on-topic tapi tanpa dasar dokumen tetap mendapat skor
               tinggi — halusinasi tidak terdeteksi. Bobot 0.4/0.4/0.2
               identik dengan Stage 2, memastikan grounding dokumen dihitung
               sama berat dengan relevansi semantik terhadap pertanyaan.

[FIX-THRESHOLD-2] HALLUCINATION_THRESHOLD DINAIKKAN dari 0.30 ke 0.50:
               Dengan bobot relevance 0.80 di v1, DAHL hampir tidak pernah
               jatuh di bawah 0.30 — threshold tidak berguna. Threshold 0.50
               sesuai skala bobot baru (0.4/0.4/0.2) dan lebih sensitif.

[FIX-GROUNDING-3] KRITERIA FLAG dim2_grounding DIPERKETAT:
               Stage 4 v1  : dim2 < 0.10 → flag (terlalu longgar)
               Stage 4 v2  : dim2 < 0.20 → flag
               ALASAN: Dengan bobot grounding yang sekarang lebih besar
               (0.40), threshold 0.10 terlalu mudah terlampaui. Threshold
               0.20 lebih sensitif menangkap respons yang tidak terkait
               konteks dokumen.

[FIX-TOKENIZER-4] TOKENISASI LEXICAL DISESUAIKAN dengan Stage 2:
               Stage 4 v1  : nltk.word_tokenize() + 17 stop-words
               Stage 4 v2  : re.findall(r'[a-zA-Z0-9]+') + 60 stop-words
                              + filter token < 4 karakter
               ALASAN: Perbedaan tokenizer dan stop-words menghasilkan skor
               lexical yang tidak sebanding antara fase training (Stage 2)
               dan evaluasi (Stage 4). Konsistensi memastikan interpretasi
               metrik yang valid.

[FIX-DIM1-5]  KRITERIA FLAG dim1_relevance DINAIKKAN dari 0.30 ke 0.35:
               Sejalan dengan threshold DAHL yang lebih ketat. Model yang
               sudah di-DPO seharusnya selalu relevan; skor relevance di
               bawah 0.35 mengindikasikan respons yang benar-benar off-topic
               atau kosong.

=============================================================================
KONSISTENSI YANG DIPERTAHANKAN DARI v1 (tidak diubah)
=============================================================================

[KEEP-1] ColPali: ColPaliForRetrieval (transformers native, bukan colpali_engine)
[KEEP-2] Query encoding: processor(text=[...], return_tensors='pt')
[KEEP-3] Tidak ada normalisasi L2 manual (konsisten dengan Stage 1 & indexing)
[KEEP-4] Full multi-vector ke Qdrant (tidak di-mean-pool)
[KEEP-5] precomputed_contexts: dict lengkap {filename, pdf_name, page_number, text}
[KEEP-6] Sentence model: paraphrase-multilingual-MiniLM-L12-v2 (CPU)
[KEEP-7] TOP_K = 3, MAX_CONTEXT_CHARS_PER_SOURCE = 1200

=============================================================================
FORMULA DAHL v2 (disesuaikan)
=============================================================================

  DAHL = 0.40 x Dim1_Relevance  +  0.40 x Dim2_Grounding  +  0.20 x Dim3_Lexical

  Dim1_Relevance  = max(0, cosine(e_response, e_query))     -- on-topic?
  Dim2_Grounding  = max(0, cosine(e_response, e_context))   -- anti-halusinasi
  Dim3_Lexical    = |T_resp ∩ T_ctx| / |T_resp|             -- keyword overlap

  Kriteria flag halusinasi (ANY terpenuhi = flagged):
    - DAHL < 0.50
    - dim1_relevance < 0.35
    - dim2_grounding < 0.20  (ada konteks)
    - response kosong
"""
print("Stage 4 v2 — dokumentasi dimuat.")

# =========================================================
# CELL 1 — INSTALL DEPENDENCY  (FAILSAFE VERSION)
# Perubahan vs versi lama:
#  - torch TIDAK lagi di-uninstall lebih dulu. Jika torch Colab bawaan
#    sudah >= 2.4 dan CUDA aktif, instalasi torch di-SKIP (penyebab
#    CalledProcessError: index download.pytorch.org / wheel 2.4.0 gagal).
#  - Jika perlu install torch: rantai fallback
#       (1) pin 2.4.0 via index cu121
#       (2) pin 2.4.0 via PyPI biasa
#       (3) torch terbaru via PyPI
#  - Semua langkah non-fatal + retry; paket ber-pin yang gagal akan
#    dicoba lagi tanpa pin versi.
#  - Verifikasi akhir (import check) di subprocess.
# colpali-engine tetap dihapus; transformers>=4.49.0 wajib
# agar ColPaliForRetrieval tersedia.
# =========================================================
import subprocess, sys, re, time

PIP = [sys.executable, "-m", "pip"]
_FAILED = []          # ringkasan paket yang gagal total


def _run(cmd, timeout=1500):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except Exception as e:                       # timeout, dll.
        return False, f"{type(e).__name__}: {e}"


def pip_install(pkgs, extra=(), retries=2, label=None):
    """Install satu grup paket. Return True jika sukses. Tidak pernah raise."""
    label = label or " ".join(pkgs)[:70]
    cmd = PIP + ["install", "-q", "--disable-pip-version-check", *pkgs, *extra]
    for attempt in range(1, retries + 1):
        ok, out = _run(cmd)
        if ok:
            print(f"  ✅ {label}")
            return True
        print(f"  ⚠️  {label} — gagal (percobaan {attempt}/{retries})")
        time.sleep(3)
    tail = "\n".join(out.strip().splitlines()[-2:])[:300]
    print(f"     ↳ {tail}")
    return False


def _unpin(spec):
    """'pandas==2.2.2' -> 'pandas', 'nltk>=3.9.1' -> 'nltk'"""
    return re.split(r"[<>=!~;\[ ]", spec, maxsplit=1)[0]


def pip_group(pkgs, label):
    """Coba semua sekaligus -> per paket (ber-pin) -> per paket tanpa pin."""
    if pip_install(pkgs, label=label):
        return True
    print(f"  ↻ {label}: fallback install per-paket")
    all_ok = True
    for p in pkgs:
        if pip_install([p], retries=1, label=p):
            continue
        bare = _unpin(p)
        if bare != p and pip_install([bare], retries=1, label=f"{bare} (tanpa pin)"):
            continue
        _FAILED.append(p)
        all_ok = False
    return all_ok


def torch_status():
    """Cek torch di SUBPROCESS (tidak meng-import torch ke kernel ini)."""
    ok, out = _run([sys.executable, "-c",
        "import torch;print(torch.__version__.split('+')[0], torch.cuda.is_available())"], timeout=300)
    if not ok:
        return None, False
    try:
        ver, cuda = out.strip().splitlines()[-1].split()
        return ver, cuda == "True"
    except Exception:
        return None, False


def _ver_tuple(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:2])


# ---------------------------------------------------------
# 1) Bersihkan paket NON-torch (torch sengaja tidak disentuh)
# ---------------------------------------------------------
print("🧹 Uninstall paket lama (torch dibiarkan)...")
_run(PIP + ["uninstall", "-y", "transformers", "accelerate", "bitsandbytes",
            "peft", "colpali-engine"])

# ---------------------------------------------------------
# 2) numpy (best-effort; jika gagal pakai numpy bawaan)
# ---------------------------------------------------------
print("📦 numpy")
if not pip_install(["numpy==1.26.4"], retries=2):
    print("  ⚠️  Lanjut dengan numpy bawaan runtime.")

# ---------------------------------------------------------
# 3) torch — skip jika sudah OK, kalau tidak: rantai fallback
# ---------------------------------------------------------
print("🔥 torch")
ver, cuda = torch_status()
if ver and cuda and _ver_tuple(ver) >= (2, 4):
    print(f"  ✅ torch {ver} (CUDA aktif) sudah terpasang — instalasi torch DILEWATI.")
else:
    print(f"  torch saat ini: {ver or 'tidak ada'} (CUDA={cuda}) → install ulang")
    TV = ["torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0"]
    chain = [
        ("torch 2.4.0 via index cu121",
         TV, ["--index-url", "https://download.pytorch.org/whl/cu121"]),
        ("torch 2.4.0 via PyPI",
         TV, []),
        ("torch terbaru via PyPI",
         ["torch", "torchvision", "torchaudio"], []),
    ]
    for name, pk, ex in chain:
        print(f"  → mencoba: {name}")
        if pip_install(pk, extra=ex, retries=2, label=name):
            ver, cuda = torch_status()
            if ver:
                break
    ver, cuda = torch_status()
    if not ver:
        raise RuntimeError(
            "❌ torch gagal terpasang lewat semua jalur. Pastikan koneksi internet "
            "Colab aktif, lalu Runtime → Disconnect and delete runtime, jalankan ulang.")
    print(f"  ✅ torch {ver} terpasang (CUDA={cuda})")

# ---------------------------------------------------------
# 4) Library lain (tiap grup punya fallback per-paket / tanpa pin)
# ---------------------------------------------------------
print("📦 HF stack")
pip_group([
    "transformers>=4.49.0",
    "accelerate>=0.34.2",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.3",
    "datasets>=2.20.0,<3.0.0",
    "sentencepiece",
    "protobuf",
], "HF stack")

print("📦 Eval / retrieval stack")
pip_group([
    "pandas==2.2.2",
    "scikit-learn==1.5.1",
    "rouge-score==0.1.2",
    "bert-score==0.3.13",
    "sentence-transformers>=2.2.2,<4.0.0",
    "nltk>=3.9.1",  # [BUGFIX-NLG-5] 3.8.1 tidak kompatibel dengan Python 3.12
    "matplotlib",
    "seaborn",
    "qdrant-client>=1.10.0,<2.0.0",
    "scipy",
    "statsmodels",   # dipakai CELL 21 (McNemar)
], "Eval stack")

print("📦 PDF / OCR")
pip_group(["pymupdf", "pytesseract", "Pillow", "psutil"], "PDF/OCR")

ok, _ = _run(["apt-get", "install", "-q", "-y", "tesseract-ocr", "tesseract-ocr-ind"], timeout=600)
print("  ✅ tesseract" if ok else "  ⚠️  tesseract gagal (OCR opsional, dilanjutkan)")

# ---------------------------------------------------------
# 5) Verifikasi akhir
# ---------------------------------------------------------
print("\n🔎 Verifikasi import...")
CHECKS = {
    "torch":         "import torch",
    "transformers":  "from transformers import ColPaliForRetrieval, ColPaliProcessor, BitsAndBytesConfig",
    "peft":          "import peft",
    "bitsandbytes":  "import bitsandbytes",
    "datasets":      "import datasets",
    "sentence-tf":   "import sentence_transformers",
    "qdrant":        "import qdrant_client",
    "rouge/bert":    "import rouge_score, bert_score",
    "pymupdf":       "import fitz",
}
_bad = []
for k, stmt in CHECKS.items():
    ok, out = _run([sys.executable, "-c", stmt], timeout=300)
    print(f"  {'✅' if ok else '❌'} {k}")
    if not ok:
        _bad.append((k, out.strip().splitlines()[-1] if out.strip() else ""))

if _bad or _FAILED:
    print("\n⚠️  Ada masalah:")
    for k, e in _bad:
        print(f"   - import {k}: {e}")
    for p in _FAILED:
        print(f"   - gagal install: {p}")
    print("   Coba: Runtime → Restart session, lalu jalankan CELL 1 sekali lagi.")
else:
    print("\n✅ Semua dependency OK.")

print("⚠️  RESTART SESSION (Runtime → Restart session) jika ini instalasi pertama, "
      "lalu lanjut ke CELL 2 tanpa menjalankan CELL 1 lagi.")


In [ ]:
# =========================================================
# CELL 2 — SETUP & IMPORTS
# =========================================================
import os
import re
import gc
import io
import json
import fitz        # PyMuPDF
import warnings
import logging
import subprocess

import nltk
import psutil
from PIL import Image

import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    ColPaliForRetrieval,   # [KEEP-1] native transformers, bukan colpali_engine
    ColPaliProcessor,
)
from peft import PeftModel
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import bert_score
from sentence_transformers import SentenceTransformer, util
from qdrant_client import QdrantClient
from google.colab import drive, userdata

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# punkt wajib untuk NLTK >= 3.8.1 (dipakai di compute_nlg_metrics)
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# [FIX] import matplotlib global, dipakai oleh beberapa plot di bawah
# (ablation plot, NLI plot) yang sebelumnya cuma punya import lokal
# di dalam fungsi lain -> NameError: name 'plt' is not defined.
import matplotlib.pyplot as plt

# [FIX] Konstanta warna plot juga sebelumnya cuma didefinisikan LOKAL
# di dalam fungsi plot_comparison() -> NameError: name 'C_BASE' is not
# defined saat dipakai di plot ablation/NLI yang ada di scope modul.
C_BASE = "#4C9BE8"
C_DPO  = "#E8834C"
C_RED  = "#E84C4C"
C_GRN  = "#4CB87A"

# =========================================================
# CELL 3 — CEK GPU
# =========================================================
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
    print("VRAM Total     :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    raise RuntimeError("❌ GPU tidak aktif!")

# =========================================================
# CELL 4 — MOUNT DRIVE & KONFIGURASI
# =========================================================
def robust_mount():
    mount_path = "/content/drive"
    if not os.path.exists(os.path.join(mount_path, "MyDrive")):
        drive.mount(mount_path, force_remount=True)
        print("✅ Google Drive di-mount.")
    else:
        print("✅ Google Drive sudah ter-mount.")

robust_mount()

# ---- Path -----------------------------------------------
LLM_MODEL_ID      = "Qwen/Qwen2-7B-Instruct"
DPO_ADAPTER_PATH  = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/Dataset/qwen2-7b-dpo-lora-adapter"
IMAGE_DIR         = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/Dataset/extracted_images"
PDF_SOURCE_FOLDER = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/Dataset"
OUTPUT_DIR_EVAL   = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/OUTPUT_EVAL"
os.makedirs(OUTPUT_DIR_EVAL, exist_ok=True)

# =========================================================
# [MULTI-EPOCH] Evaluasi TPS diulang N_EPOCHS kali (dengan sampling
# stokastik yang sama, do_sample=True) supaya hasil (avg DAHL,
# hallucination rate, dst.) yang dilaporkan bukan kebetulan dari satu
# kali run, melainkan konsisten lintas repetisi (dilaporkan sebagai
# mean ± std pada CELL 18B, di akhir loop epoch).
#
# Semua checkpoint/output dari sini disimpan di OUTPUT_DIR_EVAL yang
# SAMA (folder Drive yang di-share) — supaya proses bisa dilanjut dari
# akun Google/Colab manapun, asal folder tsb sudah ditambahkan sebagai
# shortcut ("Add shortcut to Drive") di akun tersebut sehingga path-nya
# identik: My Drive/Thesis-Agung_Prawoto-56225110019/OUTPUT_EVAL.
# =========================================================
N_EPOCHS   = 3
EPOCH_LIST = list(range(1, N_EPOCHS + 1))

# ---- Secrets --------------------------------------------
QDRANT_URL     = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
HF_TOKEN       = userdata.get("HF_TOKEN")

# ---- ColPali & Qdrant -----------------------------------
COLPALI_ID      = "vidore/colpali-v1.3-hf"   # [KEEP-1] checkpoint merged, tanpa adapter LoRA
COLLECTION_NAME = "colpali_v13_hf_multivec_rag_index"
TOP_K_CONTEXT   = 3                           # [KEEP-7]

# ---- LLM generation hyperparameter ---------------------
MAX_NEW_TOKENS            = 200
TEMPERATURE               = 0.4
TOP_P                     = 0.9
REPETITION_PENALTY        = 1.1
MAX_CONTEXT_CHARS_PER_SOURCE = 1200           # [KEEP-7]

# ---- Dataset eval ---------------------------------------
EVAL_CSV_URL = (
    "<PUBLISHED_SHEET_CSV_URL>"
)

# =========================================================
# [FIX-WEIGHT-1] BOBOT DAHL — disesuaikan dengan Stage 2
#
# Stage 2 S_final  : 0.4 x Sem_grounding + 0.4 x Sem_relevance + 0.2 x Lexical
# Stage 4 v1 DAHL  : 0.80 x Dim1 + 0.10 x Dim2 + 0.10 x Dim3   <-- TIDAK KONSISTEN
# Stage 4 v2 DAHL  : 0.40 x Dim1 + 0.40 x Dim2 + 0.20 x Dim3   <-- DISESUAIKAN
#
# Perubahan ini memastikan:
#   - Dim2 (grounding) memiliki bobot sama dengan Dim1 (relevance).
#   - Respons yang relevan tapi tidak berdasar dokumen tidak bisa
#     lolos dengan skor tinggi hanya karena Dim1 dominan.
# =========================================================
DAHL_W1 = 0.40   # Dim1: Semantic Relevance  (response vs query)
DAHL_W2 = 0.40   # Dim2: Semantic Grounding  (response vs context dokumen)
DAHL_W3 = 0.20   # Dim3: Lexical Overlap     (keyword resp ∩ context)

# =========================================================
# [FIX-THRESHOLD-2] HALLUCINATION_THRESHOLD — dinaikkan ke 0.50
#
# Stage 4 v1 : 0.30 — terlalu rendah, hampir tidak pernah terlampaui
#              dengan bobot v1 (W1=0.80 membuat skor selalu tinggi)
# Stage 4 v2 : 0.50 — sesuai dengan skala bobot baru (0.4/0.4/0.2)
#              dan lebih sensitif menangkap respons berkualitas rendah
# =========================================================
HALLUCINATION_THRESHOLD = 0.50

# =========================================================
# [FIX-TOKENIZER-4] STOP-WORDS — disesuaikan dengan Stage 2 (89 kata)
#
# Stage 4 v1 : 17 stop-words + filter <= 1 karakter + nltk.word_tokenize()
# Stage 4 v2 : 89 stop-words + filter < 4 karakter + re.findall()
#              identik dengan STOP_WORDS_ID dan tokenize_for_lexical() Stage 2
# =========================================================
STOP_WORDS_ID: set = {
    # --- stop-words dari Stage 2 (lengkap, tidak diubah) ---
    "yang", "dan", "di", "ini", "itu", "dengan", "untuk", "dari", "ke",
    "ada", "tidak", "pada", "adalah", "dalam", "apa", "bagaimana", "berapa",
    "siapa", "kapan", "dimana", "mengapa", "apakah", "juga", "sudah", "akan",
    "bisa", "harus", "boleh", "saya", "anda", "kamu", "kami", "kita", "mereka",
    "dia", "ia", "lebih", "sangat", "saja", "jika", "maka", "atau", "tetapi",
    "namun", "karena", "oleh", "setelah", "sebelum", "ketika", "saat", "hingga",
    "sampai", "melalui", "tentang", "antara", "mengenai", "tersebut", "bahwa",
    "hal", "cara", "serta", "adapun", "seperti", "sebagai", "secara", "telah",
    "sedang", "perlu", "dapat", "sesuai", "atas", "bawah", "tanpa", "setiap",
    "semua", "seluruh",
    # --- tambahan function words umum Bahasa Indonesia ----
    "nya", "pun", "lah", "kah", "pula", "agar", "supaya", "bagi",
    "bila", "sehingga", "sejak", "yaitu", "yakni",
}
MIN_TOKEN_LEN = 4   # [FIX-TOKENIZER-4] filter token < 4 karakter (identik Stage 2)

PAGE_PATTERN = re.compile(r"_p(\d{4})\.png$", re.IGNORECASE)

print(f"LLM model           : {LLM_MODEL_ID}")
print(f"ColPali model       : {COLPALI_ID}")
print(f"Collection          : {COLLECTION_NAME}")
print(f"\n--- Konfigurasi DAHL v2 (disesuaikan dengan Stage 2) ---")
print(f"  Bobot W1 (relevance)  : {DAHL_W1}  [v1 was 0.80]")
print(f"  Bobot W2 (grounding)  : {DAHL_W2}  [v1 was 0.10]")
print(f"  Bobot W3 (lexical)    : {DAHL_W3}  [v1 was 0.10]")
print(f"  Halusinasi threshold  : {HALLUCINATION_THRESHOLD}  [v1 was 0.30]")
print(f"  Stop-words count      : {len(STOP_WORDS_ID)}  [v1 had 17]")
print(f"  Min token length      : {MIN_TOKEN_LEN}  [v1 had 1]")

# =========================================================
# CELL 5 — LOAD EVALUATION DATASET
# Identik dengan v1: mapping kolom CSV → nama standar.
# =========================================================
print("\nLoading evaluation dataset dari Google Sheets CSV...")
df_eval = pd.read_csv(EVAL_CSV_URL)
df_eval.columns = [c.strip() for c in df_eval.columns]

COLUMN_MAP = {
    "kode pertanyaan"       : "kode",
    "topik"                 : "topik",
    "pertanyaan"            : "prompt",
    "sumber file"           : "sumber_file",
    "jawaban (ground truth)": "reference",
}
rename_dict = {col: COLUMN_MAP[col.strip().lower()]
               for col in df_eval.columns if col.strip().lower() in COLUMN_MAP}
df_eval = df_eval.rename(columns=rename_dict)

if "prompt" not in df_eval.columns:
    raise ValueError(
        f"❌ Kolom 'Pertanyaan' tidak ditemukan setelah mapping. "
        f"Kolom asli di CSV: {list(pd.read_csv(EVAL_CSV_URL).columns)}"
    )

if "reference" not in df_eval.columns:
    df_eval["reference"] = ""
for optional_col in ("kode", "topik", "sumber_file"):
    if optional_col not in df_eval.columns:
        df_eval[optional_col] = ""

df_eval["prompt"]    = df_eval["prompt"].fillna("").astype(str)
df_eval["reference"] = df_eval["reference"].fillna("").astype(str)
df_eval = df_eval[df_eval["prompt"].str.strip() != ""].reset_index(drop=True)

dataset = Dataset.from_pandas(df_eval)
HAS_REFERENCE = any(r.strip() for r in df_eval["reference"])

print(f"✅ Dataset dimuat: {len(dataset)} baris query.")
print(f"   Kolom terpakai             : {list(df_eval.columns)}")
print(f"   Ground-truth tersedia      : {HAS_REFERENCE}")

# =========================================================
# CELL 6 — HELPER FUNCTIONS
# =========================================================

def clean_mem():
    """Bersihkan memori GPU/CPU di antara tahapan pipeline."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


# =========================================================
# [MULTI-EPOCH / FAILSAFE] Atomic JSON save & safe JSON load
# =========================================================
# Dipakai untuk SEMUA checkpoint & output JSON di seluruh notebook ini
# (baseline/dpo checkpoint per-epoch, evaluation_detail per-epoch,
# epoch_summaries_progress.json, dll).
#
# Kenapa perlu "atomic":
#   Google Colab bisa mati/disconnect KAPAN SAJA (limit 5j30m, idle
#   timeout, crash, dsb). Kalau proses mati persis saat json.dump()
#   sedang menulis file, file checkpoint akan CORRUPT (JSON setengah
#   jadi) dan tidak bisa dibaca lagi saat resume -> semua progres hilang.
#
# Caranya:
#   1. Tulis dulu ke file sementara "<path>.tmp".
#   2. Kalau file lama ("<path>") sudah ada, pindahkan dulu jadi
#      "<path>.bak" (backup 1 versi sebelumnya).
#   3. os.replace(tmp, path) -- ini operasi atomic di level OS (POSIX/
#      NTFS), jadi file "path" TIDAK PERNAH dalam keadaan setengah tertulis.
#
# safe_json_load() akan otomatis fallback ke file ".bak" kalau file utama
# gagal dibaca (corrupt/kosong) -- jadi paling apes cuma kehilangan 1
# siklus save_every, bukan seluruh checkpoint.
# =========================================================
def atomic_json_save(path: str, obj) -> bool:
    """Simpan `obj` sebagai JSON ke `path` secara atomic + backup rotasi."""
    tmp_path = path + ".tmp"
    bak_path = path + ".bak"
    try:
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        if os.path.exists(path):
            try:
                os.replace(path, bak_path)
            except Exception:
                pass  # backup gagal bukan fatal, lanjut tetap simpan yang baru
        os.replace(tmp_path, path)
        return True
    except Exception as e:
        print(f"⚠️ atomic_json_save gagal untuk {path}: {e}")
        try:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
        except Exception:
            pass
        return False


def safe_json_load(path: str, default=None):
    """Baca JSON dari `path`; kalau corrupt/hilang, coba `path + '.bak'`.
    Kalau keduanya gagal, kembalikan `default` (bukan raise error) —
    supaya notebook tidak berhenti total gara-gara satu file rusak."""
    for candidate in (path, path + ".bak"):
        if os.path.exists(candidate):
            try:
                with open(candidate, "r", encoding="utf-8") as f:
                    data = json.load(f)
                if candidate.endswith(".bak"):
                    print(f"⚠️ File utama {path} gagal dibaca — berhasil pulih dari backup {candidate}.")
                return data
            except Exception as e:
                print(f"⚠️ Gagal baca {candidate}: {e}")
                continue
    return default


def print_mem_usage(label: str = ""):
    import psutil
    tag = f" [{label}]" if label else ""
    ram = psutil.virtual_memory()
    print(f"  [MEM]{tag} RAM : {ram.used/1024**3:.2f} / {ram.total/1024**3:.2f} GB ({ram.percent:.0f}%)")
    if torch.cuda.is_available():
        alloc    = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved()  / 1024**3
        total    = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  [MEM]{tag} VRAM: alloc={alloc:.2f}GB reserved={reserved:.2f}GB / total={total:.2f}GB")


def tokenize_for_lexical(text: str) -> set:
    """
    [FIX-TOKENIZER-4] Tokenisasi lexical — identik dengan Stage 2.

    Stage 4 v1 menggunakan nltk.word_tokenize() + 17 stop-words.
    Stage 4 v2 menggunakan re.findall() + 60 stop-words + filter
    token < 4 karakter — persis sama dengan tokenize_for_lexical()
    di Stage 2, sehingga skor Dim3 (Lexical Overlap) dapat dibandingkan
    secara valid antara fase training dan evaluasi.
    """
    tokens = re.findall(r"[a-zA-Z0-9]+", text.lower())
    return {
        t for t in tokens
        if len(t) >= MIN_TOKEN_LEN and t not in STOP_WORDS_ID
    }


def parse_page_number(image_filename: str):
    """Ekstrak page_number 0-indexed dari nama file gambar (e.g. _p0003.png -> 3)."""
    m = PAGE_PATTERN.search(image_filename or "")
    return int(m.group(1)) if m else None


def get_page_text_from_pdf(pdf_name: str, page_number: int) -> str:
    """Ambil teks asli halaman dari file PDF via PyMuPDF."""
    if not pdf_name or page_number is None:
        return ""
    pdf_path = os.path.join(PDF_SOURCE_FOLDER, pdf_name + ".pdf")
    if not os.path.exists(pdf_path):
        return ""
    try:
        doc  = fitz.open(pdf_path)
        page = doc[page_number]
        text = page.get_text("text").strip()
        doc.close()
        return text
    except Exception as e:
        print(f"  [WARN] Gagal baca {pdf_name}.pdf hal.{page_number}: {e}")
        return ""


def get_page_text_with_ocr_fallback(
    pdf_name: str, page_number: int, image_path: str
) -> str:
    """Teks dari PDF; jika kosong, fallback ke OCR Tesseract dari gambar halaman.

    [FIX-DECOMPRESSION-BOMB] PIL menolak gambar > 178 MP sebagai keamanan.
    Halaman PDF beresolusi tinggi (mis. 30000×35000 px) bisa melampaui batas ini.
    Solusi:
      1. Nonaktifkan batas PIL sementara (Image.MAX_IMAGE_PIXELS = None).
      2. Buka gambar dalam mode lazy (.draft) jika memungkinkan.
      3. Resize agresif ke max 1600 px sebelum OCR — Tesseract tidak butuh
         resolusi > ~300 DPI untuk akurasi optimal; gambar lebih kecil
         justru lebih cepat dan tidak OOM.
      4. Tangkap DecompressionBombError sebagai fallback terakhir.
    """
    text = get_page_text_from_pdf(pdf_name, page_number)
    if text:
        return text
    if image_path and os.path.exists(image_path):
        try:
            import pytesseract
            from PIL import Image as _PILImage, UnidentifiedImageError

            # [FIX] Disable PIL decompression bomb limit temporarily.
            # Large scanned PDFs can exceed 178 MP. We control the risk
            # ourselves by resizing aggressively before any pixel operation.
            _orig_limit = _PILImage.MAX_IMAGE_PIXELS
            _PILImage.MAX_IMAGE_PIXELS = None  # disable check; we resize manually

            try:
                img = _PILImage.open(image_path)
                # Resize BEFORE converting to RGB to avoid allocating the
                # full giant tensor (thumbnail operates on compressed data
                # for many formats).
                OCR_MAX_PX = 1600  # Tesseract optimal range; larger = slower + OOM
                if max(img.size) > OCR_MAX_PX:
                    img.thumbnail((OCR_MAX_PX, OCR_MAX_PX), _PILImage.Resampling.LANCZOS)
                img = img.convert("RGB")
                result = pytesseract.image_to_string(img, lang="ind+eng").strip()
                img.close()
                return result
            except (OSError, UnidentifiedImageError) as e:
                print(f"  [WARN] Gambar tidak bisa dibaca ({image_path}): {e}")
                return ""
            except Exception as e:
                print(f"  [WARN] OCR gagal ({image_path}): {e}")
                return ""
            finally:
                _PILImage.MAX_IMAGE_PIXELS = _orig_limit  # restore original limit
        except ImportError:
            print("  [WARN] pytesseract belum terinstall.")
    return ""

# =========================================================
# CELL 7 — LOAD COLPALI (native transformers) + QDRANT
#          PRE-COMPUTE ALL CONTEXTS
#
# Seluruh blok ini identik dengan Stage 4 v1 [KEEP-1 s.d. KEEP-5].
# Tidak ada perubahan pada retrieval pipeline — perbedaan ada di
# DAHL scoring (Cell 9) dan hallucination audit (Cell 12).
# =========================================================
print("\nLoading ColPali (native transformers) for context retrieval...")
print_mem_usage("sebelum load ColPali")

colpali_processor = ColPaliProcessor.from_pretrained(COLPALI_ID, token=HF_TOKEN)
colpali_model     = ColPaliForRetrieval.from_pretrained(
    COLPALI_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
).eval()

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
info   = qdrant.get_collection(COLLECTION_NAME)
print(f"✅ ColPali + Qdrant siap. Collection '{COLLECTION_NAME}': {info.points_count} points")
print_mem_usage("setelah load ColPali")


def encode_query_multivector(query: str):
    """
    Encode query menjadi multi-vector (num_tokens × dim).

    [KEEP-2] Pakai processor(text=[...]) — bukan process_queries()
    [KEEP-2] Output dari output.embeddings
    [KEEP-3] Tidak ada normalisasi L2 manual
    """
    with torch.no_grad():
        inputs = colpali_processor(
            text=[query], return_tensors="pt"
        ).to(colpali_model.device)
        output = colpali_model(**inputs)
        q_vecs = output.embeddings[0].float().cpu()
    del inputs, output
    return q_vecs.numpy()   # shape: (num_tokens, dim)


def get_context_for_query(query: str):
    """
    Retrieve top-K halaman dari Qdrant menggunakan full multi-vector ColPali.

    [KEEP-4] Full multi-vector dikirim ke Qdrant — tidak di-mean-pool
    [KEEP-5] Return list dict: {filename, pdf_name, page_number, text, score}
    """
    q_vecs    = encode_query_multivector(query)
    q_multivec = q_vecs.tolist()   # list[list[float]], shape (num_tokens, dim)

    res = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_multivec,
        limit=TOP_K_CONTEXT * 2,
        with_payload=True,
    ).points

    contexts, seen = [], set()
    for hit in res:
        if len(contexts) >= TOP_K_CONTEXT:
            break
        payload  = hit.payload or {}
        filename = payload.get("file", "")
        if not filename or filename in seen:
            continue
        seen.add(filename)

        pdf_name    = payload.get("pdf_name", "")
        page_number = parse_page_number(filename)
        image_path  = os.path.join(IMAGE_DIR, filename) if filename else None
        text        = get_page_text_with_ocr_fallback(pdf_name, page_number, image_path)
        text        = text[:MAX_CONTEXT_CHARS_PER_SOURCE] if text else ""

        contexts.append({
            "filename"   : filename,
            "pdf_name"   : pdf_name,
            "page_number": page_number,
            "image_path" : image_path,
            "text"       : text,
            "score"      : hit.score,
        })

    del q_vecs
    return contexts


# [FIX-SKIP-RECOMPUTE] Jika precomputed_contexts.json sudah ada dan jumlah
# sampelnya sesuai dataset, langsung load — tidak perlu re-retrieve ulang
# (retrieval 200 query via ColPali bisa makan 15-30 menit + GPU memory).
precomputed_json_path = os.path.join(OUTPUT_DIR_EVAL, "precomputed_contexts.json")
_need_recompute = True

if os.path.exists(precomputed_json_path):
    print(f"\n📂 Ditemukan precomputed_contexts.json — mencoba load cache...")
    try:
        with open(precomputed_json_path, "r", encoding="utf-8") as f:
            _cached = json.load(f)
        if len(_cached) == len(dataset):
            all_contexts = _cached
            _need_recompute = False
            print(f"✅ Cache valid! {len(all_contexts)} contexts di-load dari file (skip retrieval).")
        else:
            print(f"  [WARN] Cache berisi {len(_cached)} entri tapi dataset berisi {len(dataset)} — re-compute.")
    except Exception as e:
        print(f"  [WARN] Gagal load cache: {e} — re-compute.")

if _need_recompute:
    print("\nPre-computing full contexts untuk semua query...")
    all_contexts = []
    for i, example in enumerate(dataset):
        ctxs = get_context_for_query(example["prompt"])
        all_contexts.append(ctxs)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(dataset)} query diproses...")
    print(f"✅ Semua {len(all_contexts)} contexts berhasil di-retrieve.")

    # Simpan / update cache
    with open(precomputed_json_path, "w", encoding="utf-8") as f:
        json.dump(all_contexts, f, ensure_ascii=False, indent=2)
    print(f"💾 Cache disimpan ke {precomputed_json_path}")

del colpali_model
del colpali_processor
clean_mem()
print_mem_usage("setelah unload ColPali")
print("✅ ColPali berhasil dihapus dari VRAM.")

# =========================================================
# CELL 8 — LOAD SENTENCE TRANSFORMER (DAHL scoring)
# [KEEP-6] Model identik dengan Stage 2: paraphrase-multilingual-MiniLM-L12-v2
# =========================================================
print("\nLoading SentenceTransformer untuk DAHL scoring...")
embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu"
)
print("✅ SentenceTransformer siap.")

# =========================================================
# CELL 9 — DAHL SCORING FUNCTION (v2 — DISESUAIKAN)
#
# Formula:
#   DAHL = 0.40 x Dim1_Relevance + 0.40 x Dim2_Grounding + 0.20 x Dim3_Lexical
#
# Dim1_Relevance = max(0, cosine(e_response, e_query))
# Dim2_Grounding = max(0, cosine(e_response, e_context))
# Dim3_Lexical   = |T_resp ∩ T_ctx| / |T_resp|
#
# [FIX-WEIGHT-1] : Bobot W1=0.40, W2=0.40, W3=0.20 (identik Stage 2)
# [FIX-TOKENIZER-4]: tokenize_for_lexical() dipakai untuk Dim3, bukan
#                    tokenize_clean() dengan nltk seperti di v1
# [KEEP-5]        : contexts berisi dict dengan key 'text' (teks asli PDF/OCR)
# =========================================================
def calculate_DAHL_score(response: str, query: str, contexts: list):
    """
    Hitung skor DAHL v2 dengan bobot yang disesuaikan dengan Stage 2.

    Parameters
    ----------
    response : str
        Teks jawaban yang dihasilkan model.
    query : str
        Teks pertanyaan asli.
    contexts : list[dict]
        List dict hasil retrieval, setiap dict memiliki key 'text'
        berisi teks asli halaman dokumen (PDF/OCR).

    Returns
    -------
    tuple(float, dict)
        (dahl_score, detail_dict) — detail berisi tiga dimensi terpisah.

    Perubahan vs v1
    ---------------
    - Bobot W1 turun dari 0.80 → 0.40 [FIX-WEIGHT-1]
    - Bobot W2 naik dari 0.10 → 0.40  [FIX-WEIGHT-1]
    - Bobot W3 naik dari 0.10 → 0.20  [FIX-WEIGHT-1]
    - Dim3 menggunakan tokenize_for_lexical() (60 stopwords, MIN_TOKEN_LEN=4)
      bukan tokenize_clean() (17 stopwords, min length=1) [FIX-TOKENIZER-4]
    """
    if not response or len(response.strip()) < 5:
        return 0.0, {}

    embs = embed_model.encode(
        [response, query], convert_to_tensor=True, normalize_embeddings=True
    )

    # Dim1 — Semantic Relevance (response vs query)
    dim1 = max(0.0, util.cos_sim(embs[0], embs[1]).item())

    # Dim2 — Semantic Grounding & Dim3 — Lexical Overlap
    if contexts:
        proxy_text = " ".join([c["text"] for c in contexts if c.get("text")])
        if not proxy_text.strip():
            # Fallback jika semua halaman scanned tanpa text layer
            proxy_text = " ".join(
                c["filename"].replace("_", " ").split(".")[0] for c in contexts
            )

        ctx_emb = embed_model.encode(
            proxy_text, convert_to_tensor=True, normalize_embeddings=True
        )
        dim2 = max(0.0, util.cos_sim(embs[0], ctx_emb).item())

        # [FIX-TOKENIZER-4] Gunakan tokenize_for_lexical() identik Stage 2
        t_resp = tokenize_for_lexical(response)
        t_ctx  = tokenize_for_lexical(proxy_text)
        dim3   = len(t_resp & t_ctx) / len(t_resp) if t_resp else 0.0
    else:
        dim2, dim3 = 0.0, 0.0

    # [FIX-WEIGHT-1] Formula DAHL v2 — disesuaikan dengan Stage 2
    score = round(DAHL_W1 * dim1 + DAHL_W2 * dim2 + DAHL_W3 * dim3, 4)

    return score, {
        "dim1_relevance": round(dim1, 4),
        "dim2_grounding": round(dim2, 4),
        "dim3_lexical"  : round(dim3, 4),
    }

# =========================================================
# CELL 9B — DAHL SCORE: Atomic-Unit Factual Precision
# (Faithful reimplementation of Seo et al., 2024 "DAHL")
# =========================================================
#
# calculate_DAHL_score() di CELL 9 adalah proxy berbasis cosine-
# similarity + lexical overlap, dipakai untuk evaluasi performa utama
# (CELL 13/14) — TIDAK diubah oleh penambahan ini.
#
# Fungsi-fungsi di bawah mengimplementasikan DAHL Score SESUAI
# metodologi aslinya di paper Seo et al. (2024):
#
#     DAHL Score = rata-rata, untuk setiap respons, dari
#                  (jumlah atomic unit yang FAKTUAL) / (total atomic unit)
#
# di mana:
#   Step1 (Splitter) : respons dipecah menjadi "atomic unit" — kalimat
#                       yang masing-masing memuat SATU informasi yang
#                       bisa dinilai benar/salah.
#   Step2 (Checker)  : setiap atomic unit diperiksa kebenarannya secara
#                       biner (true/false) terhadap sebuah sumber
#                       pengetahuan yang kredibel.
#   Step3 (Scoring)  : unit non-committal ("tidak bisa dijawab") dan unit
#                       duplikat dibuang sebelum dihitung; lalu dihitung
#                       rasio unit benar / total unit per respons; skor
#                       akhir = rata-rata rasio tersebut di seluruh dataset.
#
# ADAPTASI UNTUK SISTEM RAG INI:
#   Sistem ini menjawab pertanyaan berdasarkan dokumen akademik yang
#   di-retrieve (ColPali + Qdrant), bukan pengetahuan dunia terbuka.
#   Maka Checker memeriksa setiap atomic unit terhadap TEKS KONTEKS YANG
#   DI-RETRIEVE itu sendiri (precomputed_contexts), bukan web/API
#   eksternal. Ini tetap setia pada arsitektur paper (atomic decomposition
#   + binary factuality check + ratio scoring), hanya mengganti sumber
#   kebenaran sesuai domain closed-book RAG.
#
# DAHL Score di sini berfungsi sebagai VALIDASI INDEPENDEN dari TPS/proxy
# lama — beda total formula (atomic-unit + LLM judgment, bukan cosine
# similarity + lexical overlap), dan lewat Fixed Checker (CELL 19), beda
# juga model yang menilai vs model yang dievaluasi.
# =========================================================

# --- Noncommittal pattern (paper men-exclude respons "tidak bisa dijawab") ---
NONCOMMITTAL_PATTERNS = [
    r"tidak (dapat|bisa) (saya )?jawab",
    r"tidak (ada|tersedia) informasi",
    r"saya tidak (tahu|mengetahui)",
    r"tidak dapat ditemukan",
    r"maaf,? saya tidak",
    r"i don'?t know",
    r"cannot be answered",
    r"no information (is )?available",
]
NONCOMMITTAL_RE = re.compile("|".join(NONCOMMITTAL_PATTERNS), re.IGNORECASE)


def is_noncommittal(unit: str) -> bool:
    return bool(NONCOMMITTAL_RE.search(unit.strip()))


# --- Splitter prompt (adaptasi Figure 9 paper) ---
SPLITTER_PROMPT_TEMPLATE = """Pecah teks di bawah menjadi kalimat-kalimat yang lebih kecil, \
masing-masing hanya memuat SATU fakta tunggal. Hindari penggunaan kata \
ganti seperti "ini", "itu", "tersebut" — sebutkan rujukannya secara \
eksplisit. Pisahkan setiap kalimat hasil dengan baris baru. Jangan \
menambahkan penomoran, penjelasan, atau komentar apa pun.

TEKS:
{response}

KALIMAT TERPISAH:"""


def split_into_atomic_units(response: str, model, tokenizer, max_new_tokens: int = 400) -> list:
    """Step1 — Splitter. Memecah respons menjadi atomic units via LLM."""
    if not response or len(response.strip()) < 5:
        return []

    prompt = SPLITTER_PROMPT_TEMPLATE.format(response=response.strip())
    messages = [{"role": "user", "content": prompt}]
    text_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text_input, return_tensors="pt", truncation=True, max_length=2048
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministik untuk Splitter (konsisten dgn paper)
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()
    del inputs, out

    units = []
    for line in raw.split("\n"):
        line = re.sub(r"^\s*[\-\*\d]+[\.\)]?\s*", "", line).strip()
        if len(line) >= 5:
            units.append(line)
    return units


def dedup_atomic_units(units: list, jaccard_threshold: float = 0.85) -> list:
    """Hapus atomic unit yang informasinya duplikat (paper: 'sentences
    containing identical information are removed')."""
    kept, kept_tokens = [], []
    for u in units:
        t = tokenize_for_lexical(u)
        is_dup = any(
            (len(t & kt) / len(t | kt) if (t | kt) else 0) >= jaccard_threshold
            for kt in kept_tokens
        )
        if not is_dup:
            kept.append(u)
            kept_tokens.append(t)
    return kept


# --- Checker (grounded ke konteks dokumen yang di-retrieve ColPali) ---
CHECKER_PROMPT_TEMPLATE = """Berikut adalah KONTEKS DOKUMEN yang menjadi satu-satunya sumber \
kebenaran. Untuk setiap STATEMENT yang diberi nomor, tentukan apakah \
statement tersebut DIDUKUNG oleh konteks (jawab "true"), DIBANTAH/TIDAK \
SESUAI dengan konteks (jawab "false"), atau TIDAK DAPAT ditentukan dari \
konteks yang ada (jawab "unknown"). Jangan gunakan pengetahuan di luar \
konteks yang diberikan. Jawab HANYA dengan format:
1. true/false/unknown
2. true/false/unknown
... (dan seterusnya, satu baris per statement, tanpa penjelasan lain)

KONTEKS DOKUMEN:
{context}

STATEMENTS:
{numbered_units}

JAWABAN:"""


def check_atomic_units_against_context(
    units: list, context_text: str, model, tokenizer, max_new_tokens: int = 300
) -> list:
    """Step2 — Checker. Cek faktualitas setiap atomic unit terhadap
    konteks dokumen yang di-retrieve."""
    if not units:
        return []
    if not context_text or not context_text.strip():
        return ["unknown"] * len(units)

    numbered = "\n".join(f"{i+1}. {u}" for i, u in enumerate(units))
    prompt = CHECKER_PROMPT_TEMPLATE.format(
        context=context_text[:4000], numbered_units=numbered
    )
    messages = [{"role": "user", "content": prompt}]
    text_input = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text_input, return_tensors="pt", truncation=True, max_length=3072
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministik untuk Checker
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()
    del inputs, out

    parsed = {}
    for line in raw.split("\n"):
        m = re.match(r"\s*(\d+)[\.\)]\s*(true|false|unknown)", line, re.IGNORECASE)
        if m:
            parsed[int(m.group(1))] = m.group(2).lower()

    # [Step3 paper] jika jumlah label tidak cocok dengan jumlah unit,
    # respons dianggap tidak bisa dinilai (discrepancy → exclude)
    if len(parsed) != len(units):
        return ["unknown"] * len(units)

    return [parsed[i + 1] for i in range(len(units))]


def calculate_dahl_score_atomic(
    response: str, contexts: list, model, tokenizer,
    checker_model=None, checker_tokenizer=None,
):
    """
    Step1–3 lengkap → DAHL Score (persis metodologi paper) untuk satu respons.

    Parameters
    ----------
    checker_model, checker_tokenizer : optional
        Kalau diisi, dipakai untuk Splitter & Checker SEBAGAI GANTI
        `model`/`tokenizer` (yang dipakai generate jawaban). Tujuannya:
        supaya model yang dievaluasi tidak menilai jawabannya sendiri —
        pakai 1 model fixed yang sama untuk cek Baseline maupun DPO
        (lihat CELL 19). Kalau None, fallback ke `model`/`tokenizer`
        (model menilai dirinya sendiri).

    Returns
    -------
    (score: float | None, details: dict)
        score None jika seluruh unit noncommittal/unknown — respons
        tersebut dikeluarkan dari rata-rata akhir, sesuai paper.
    """
    if not response or len(response.strip()) < 5:
        return None, {"reason": "empty_response", "atomic_units": [], "labels": []}

    cm = checker_model if checker_model is not None else model
    ct = checker_tokenizer if checker_tokenizer is not None else tokenizer

    context_text = (
        "\n\n".join(c["text"] for c in contexts if c.get("text")) if contexts else ""
    )

    units = split_into_atomic_units(response, cm, ct)
    units = [u for u in units if not is_noncommittal(u)]
    units = dedup_atomic_units(units)

    if not units:
        return None, {
            "reason": "no_atomic_units_after_filtering",
            "atomic_units": [],
            "labels": [],
        }

    labels = check_atomic_units_against_context(units, context_text, cm, ct)

    valid = [(u, l) for u, l in zip(units, labels) if l in ("true", "false")]
    if not valid:
        return None, {
            "reason": "all_units_unknown",
            "atomic_units": units,
            "labels": labels,
        }

    n_true = sum(1 for _, l in valid if l == "true")
    score = round(n_true / len(valid), 4)

    return score, {
        "atomic_units"  : units,
        "labels"        : labels,
        "n_total_units" : len(units),
        "n_valid_units" : len(valid),
        "n_true"        : n_true,
    }


# =========================================================
# DAHL Hallucination Audit — sejajar dengan hallucination_audit()
# lama (CELL 12), tapi berbasis DAHL Score paper (atomic-unit), bukan
# proxy embedding. Nama fungsi & konstanta dibedakan dari versi proxy
# (bukan karena beda metrik secara konsep, tapi supaya tidak collision
# kalau dipakai berdampingan).
# =========================================================
DAHL_HALLUCINATION_THRESHOLD = 0.50   # > separuh atomic unit salah/tak didukung → flag


def dahl_hallucination_audit(results: list, threshold: float = DAHL_HALLUCINATION_THRESHOLD) -> list:
    """
    Tandai setiap respons sebagai halusinasi berdasarkan DAHL Score (atomic).

    Kriteria flag (ANY terpenuhi = hallucination_flag = True):
      1. response kosong         : model gagal generate
      2. dahl_score is None      : seluruh atomic unit noncommittal/
                                    unknown/discrepancy → respons tidak
                                    bisa diverifikasi sama sekali,
                                    diperlakukan sebagai berisiko
      3. dahl_score < threshold  : mayoritas atomic unit dalam respons
                                    salah / tidak didukung konteks dokumen

    Returns
    -------
    list[dict] : hasil ditambah field 'hallucination_flag' dan
                 'hallucination_reason'.
    """
    audited = []
    for r in results:
        flags = []
        score = r.get("dahl_score")

        if not r["response"]:
            flags.append("Empty response (model gagal generate)")

        if score is None:
            reason = r.get("dahl_details", {}).get("reason", "unknown")
            flags.append(
                f"dahl_score=None (reason={reason}) — tidak ada atomic unit "
                f"yang bisa diverifikasi, diperlakukan sebagai berisiko"
            )
        elif score < threshold:
            flags.append(
                f"dahl_score={score:.3f} < threshold={threshold} "
                f"({r.get('dahl_details', {}).get('n_true', '?')}/"
                f"{r.get('dahl_details', {}).get('n_valid_units', '?')} unit faktual)"
            )

        audited.append({
            **r,
            "hallucination_flag"  : len(flags) > 0,
            "hallucination_reason": "; ".join(flags) if flags else "OK",
        })
    return audited


def aggregate_dahl_atomic(results: list, label: str = "Model") -> dict:
    """Rata-rata DAHL Score (atomic) + hallucination rate, kalau results
    sudah melewati dahl_hallucination_audit()."""
    scores = [r["dahl_score"] for r in results if r.get("dahl_score") is not None]
    n_excluded = len(results) - len(scores)
    print(f"\n[{label}] DAHL Score — valid: {len(scores)}/{len(results)} (excluded: {n_excluded})")

    agg = {"model": label, "n_samples": len(results), "n_valid": len(scores), "n_excluded": n_excluded}
    if scores:
        agg.update({
            "avg_dahl": round(float(np.mean(scores)), 4),
            "std_dahl": round(float(np.std(scores)), 4),
            "min_dahl": round(float(np.min(scores)), 4),
            "max_dahl": round(float(np.max(scores)), 4),
        })
    else:
        agg.update({"avg_dahl": None, "std_dahl": None, "min_dahl": None, "max_dahl": None})

    if results and "hallucination_flag" in results[0]:
        hall_cnt = sum(1 for r in results if r["hallucination_flag"])
        agg["hallucination_count"] = hall_cnt
        agg["hallucination_rate"]  = round(hall_cnt / len(results), 4)
        print(
            f"[{label}] Hallucination (DAHL) — flagged: {hall_cnt}/{len(results)} "
            f"({agg['hallucination_rate']:.1%}) | threshold={DAHL_HALLUCINATION_THRESHOLD}"
        )

    return agg


# =========================================================
# CELL 10 — NLG METRICS FUNCTION
# Identik dengan v1 (ROUGE, BLEU, BERTScore multilingual).
# =========================================================
def compute_nlg_metrics(generated_list, reference_list):
    """Hitung ROUGE-1/2/L, BLEU, dan BERTScore (multilingual)."""
    valid_refs = [r for r in reference_list if r and r.strip()]
    if not valid_refs:
        print("  ⚠️  Tidak ada ground-truth reference. NLG metrics dilewati.")
        return {}

    rouge  = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
    smooth = SmoothingFunction().method1

    r1_list, r2_list, rl_list, bleu_list = [], [], [], []
    gen_valid, ref_valid = [], []

    for gen, ref in zip(generated_list, reference_list):
        if not ref or not ref.strip():
            continue
        s = rouge.score(ref, gen)
        r1_list.append(s["rouge1"].fmeasure)
        r2_list.append(s["rouge2"].fmeasure)
        rl_list.append(s["rougeL"].fmeasure)
        try:
            ref_tok = [nltk.word_tokenize(ref.lower())]
            gen_tok = nltk.word_tokenize(gen.lower())
            bleu_list.append(sentence_bleu(ref_tok, gen_tok, smoothing_function=smooth))
        except Exception:
            pass
        gen_valid.append(gen)
        ref_valid.append(ref)

    print(f"  Menghitung BERTScore untuk {len(gen_valid)} pasang (multilingual)...")
    if gen_valid:
        _, _, F1 = bert_score.score(
            gen_valid,
            ref_valid,
            model_type="bert-base-multilingual-cased",
            device=str(torch.device("cuda" if torch.cuda.is_available() else "cpu")),
            verbose=False,
        )
        bert_f1 = round(F1.mean().item(), 4)
    else:
        bert_f1 = 0.0

    return {
        "ROUGE-1"     : round(np.mean(r1_list),   4) if r1_list   else 0.0,
        "ROUGE-2"     : round(np.mean(r2_list),   4) if r2_list   else 0.0,
        "ROUGE-L"     : round(np.mean(rl_list),   4) if rl_list   else 0.0,
        "BLEU"        : round(np.mean(bleu_list), 4) if bleu_list else 0.0,
        "BERTScore F1": bert_f1,
    }

# =========================================================
# CELL 11 — GENERATE RESPONSES FUNCTION
# Identik dengan v1: generate + hitung DAHL per-sampel.
# Perbedaan efektif terjadi karena calculate_DAHL_score() sudah
# menggunakan bobot baru [FIX-WEIGHT-1] dan tokenizer baru [FIX-TOKENIZER-4].
# =========================================================
def generate_responses_llm(
    model, tokenizer, dataset, precomputed_contexts, label="Model",
    temperature_override: float = None, compute_true_dahl: bool = False,
    checkpoint_path: str = None, save_every: int = 5,
    checker_model=None, checker_tokenizer=None,
):
    """
    Generate respons LLM untuk setiap query dalam dataset.

    Parameters
    ----------
    model : AutoModelForCausalLM atau PeftModel
        Model LLM yang sudah di-load (baseline atau DPO).
    tokenizer : AutoTokenizer
    dataset : datasets.Dataset
    precomputed_contexts : list[list[dict]]
        List of context lists, satu per baris dataset.
    label : str
        Label untuk logging (e.g. 'Baseline' atau 'DPO Model').
    temperature_override : float, optional
        Jika diisi, override TEMPERATURE global — dipakai untuk ablation
        temperature (CELL 19). Jika None, pakai TEMPERATURE global.
    compute_true_dahl : bool
        False (default, dipakai CELL 13/14)  : "dahl_score"/"dahl_details"
            dihitung dengan calculate_DAHL_score() — proxy/TPS lama,
            TIDAK BERUBAH dari evaluasi performa utama.
        True  (dipakai CELL 19)               : "dahl_score"/"dahl_details"
            dihitung dengan calculate_dahl_score_atomic() — DAHL Score
            PERSIS metodologi paper Seo et al. (2024), dipakai sebagai
            validasi independen yang TIDAK memakai formula TPS.
        Satu field, dua kemungkinan sumber — supaya tidak ada dua nama
        metrik paralel yang membingungkan.
    checker_model, checker_tokenizer : optional
        Model fixed untuk Splitter+Checker (lihat calculate_dahl_score_atomic,
        dipakai hanya kalau compute_true_dahl=True). Kalau None, model yang
        dievaluasi menilai jawabannya sendiri.
    checkpoint_path : str, optional
        Kalau diisi, hasil disimpan incremental ke file JSON ini setiap
        `save_every` sampel. Kalau file sudah ada saat fungsi dipanggil
        (mis. run sebelumnya disconnect di tengah jalan), sampel yang
        'kode'-nya sudah ada di file akan DI-SKIP — otomatis resume dari
        sampel terakhir yang belum selesai, tidak mulai dari nol lagi.
    save_every : int
        Frekuensi penyimpanan checkpoint (tiap N sampel baru).

    Returns
    -------
    list[dict] : satu dict per sampel, berisi response, DAHL score, dll.
    """
    gen_temperature = temperature_override if temperature_override is not None else TEMPERATURE
    mode = "DAHL Score (atomic, paper)" if compute_true_dahl else "DAHL Score (proxy/TPS)"
    print(f"\n{'='*60}")
    print(f"Generating responses: {label}  (temperature={gen_temperature}, mode={mode})")
    print(f"{'='*60}")

    # --- Resume dari checkpoint kalau ada -----------------------------
    results = []
    done_kodes = set()
    if checkpoint_path and (os.path.exists(checkpoint_path) or os.path.exists(checkpoint_path + ".bak")):
        results = safe_json_load(checkpoint_path, default=[])
        done_kodes = {r["kode"] for r in results}
        if results:
            print(
                f"📂 Checkpoint ditemukan: {len(results)} sampel sudah selesai "
                f"sebelumnya — lanjut dari sisanya (skip yang sudah ada)."
            )

    for i, example in enumerate(dataset):
        query    = example["prompt"]
        reference = example["reference"]
        kode     = example.get("kode", f"Q{i+1:03d}")
        topik    = example.get("topik", "")
        contexts = precomputed_contexts[i]

        if kode in done_kodes:
            continue  # sudah ada di checkpoint — skip, lanjut ke sampel berikutnya

        # Bangun teks konteks dari teks asli dokumen [KEEP-5]
        if contexts:
            blocks = []
            for idx_ctx, c in enumerate(contexts, start=1):
                hal = (c["page_number"] + 1) if c.get("page_number") is not None else "?"
                blocks.append(
                    f"[Sumber {idx_ctx}: {c['pdf_name']}.pdf - Halaman {hal} | skor={c['score']:.3f}]\n"
                    f"{c['text']}"
                )
            context_text = "Konteks dokumen yang relevan:\n" + "\n\n".join(blocks)
        else:
            context_text = "(Tidak ada dokumen konteks yang di-retrieve.)"

        system_prompt = (
            "Kamu adalah asisten akademik Universitas Mercu Buana. "
            "Jawablah pertanyaan mahasiswa secara akurat berdasarkan dokumen resmi akademik. "
            "Berikan jawaban yang faktual, ringkas, dan tidak berhalusinasi. "
            "Sertakan nama dokumen dan halaman yang relevan dalam jawaban.\n\n"
            + context_text
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query},
        ]

        text_input = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            text_input, return_tensors="pt", truncation=True, max_length=2048
        ).to(model.device)

        eos_ids   = [tokenizer.eos_token_id]
        im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        if im_end_id is not None and im_end_id != tokenizer.unk_token_id and im_end_id not in eos_ids:
            eos_ids.append(im_end_id)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=gen_temperature,
                top_p=TOP_P,
                repetition_penalty=REPETITION_PENALTY,
                eos_token_id=eos_ids,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_len = inputs.input_ids.shape[1]
        response  = (
            tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
            if out.shape[1] > input_len else ""
        )

        # DAHL Score — proxy/TPS lama (default) ATAU atomic paper-faithful
        # (kalau compute_true_dahl=True), tidak pernah dua-duanya sekaligus
        # supaya cuma ada SATU nama metrik "dahl_score" yang konsisten.
        if compute_true_dahl:
            dahl_score, dahl_details = calculate_dahl_score_atomic(
                response, contexts, model, tokenizer,
                checker_model=checker_model,
                checker_tokenizer=checker_tokenizer,
            )
        else:
            dahl_score, dahl_details = calculate_DAHL_score(response, query, contexts)

        results.append({
            "kode"            : kode,
            "topik"           : topik,
            "prompt"          : query,
            "reference"       : reference,
            "response"        : response,
            "filenames"       : [c["filename"] for c in contexts],
            "dahl_score"      : dahl_score,
            "dahl_details"    : dahl_details,
            "temperature_used": gen_temperature,
            "model_label"     : label,
        })

        if (i + 1) % 5 == 0 or i == 0:
            if compute_true_dahl:
                d = dahl_details or {}
                extra = f"(n_true={d.get('n_true','?')}/{d.get('n_valid_units','?')} unit faktual)"
            else:
                extra = (
                    f"(dim1={dahl_details.get('dim1_relevance',0):.3f} "
                    f"dim2={dahl_details.get('dim2_grounding',0):.3f} "
                    f"dim3={dahl_details.get('dim3_lexical',0):.3f})"
                )
            score_str = f"{dahl_score:.4f}" if dahl_score is not None else "n/a"
            print(f"  [{kode}] DAHL={score_str} {extra} | {response[:55]}...")

        del inputs, out
        clean_mem()

        # --- Simpan checkpoint incremental (atomic, failsafe) ---------
        if checkpoint_path and (len(results) % save_every == 0):
            atomic_json_save(checkpoint_path, results)

    # --- Simpan checkpoint final (memastikan semua tersimpan) --------
    if checkpoint_path:
        if atomic_json_save(checkpoint_path, results):
            print(f"💾 Checkpoint final disimpan (atomic): {checkpoint_path} ({len(results)} sampel)")

    print(f"\n✅ Selesai generating {len(results)} respons untuk {label}.")
    return results

# =========================================================
# CELL 12 — HALLUCINATION AUDIT FUNCTION (v2 — DIPERKETAT)
#
# Perubahan vs v1:
#   [FIX-THRESHOLD-2] DAHL threshold naik 0.30 → 0.50
#   [FIX-GROUNDING-3] dim2 threshold naik 0.10 → 0.20
#   [FIX-DIM1-5]      dim1 threshold naik 0.30 → 0.35
#
# Kriteria flag (ANY terpenuhi = hallucination_flag = True):
#   1. response kosong        : model gagal generate
#   2. DAHL < 0.50            : skor agregat terlalu rendah
#   3. dim1_relevance < 0.35  : respons semantik jauh dari query
#   4. dim2_grounding < 0.20  : respons tidak berkaitan dokumen (ada konteks)
# =========================================================
def hallucination_audit(results):
    """
    Hallucination Audit per-sampel dengan kriteria yang diperketat (v2).

    Perubahan threshold vs v1
    -------------------------
    | Kriteria         | v1    | v2    | Alasan perubahan                   |
    |------------------|-------|-------|------------------------------------||
    | DAHL threshold   | 0.30  | 0.50  | Threshold 0.30 terlalu rendah dengan|
    |                  |       |       | bobot baru (0.40/0.40/0.20);        |
    |                  |       |       | hampir tidak pernah terlampaui     |
    | dim1_relevance   | 0.30  | 0.35  | Sejalan dengan threshold ketat baru |
    | dim2_grounding   | 0.10  | 0.20  | Bobot grounding yang lebih tinggi   |
    |                  |       |       | memerlukan threshold yang lebih     |
    |                  |       |       | sensitif untuk deteksi non-grounded |

    Returns
    -------
    list[dict] : hasil ditambah field 'hallucination_flag' dan 'hallucination_reason'.
    """
    audited = []
    for r in results:
        flags   = []
        dahl    = r["dahl_score"]
        details = r.get("dahl_details", {})

        if not r["response"]:
            flags.append("Empty response (model gagal generate)")

        if dahl is None:
            flags.append("dahl_score=None (response kosong/no-atomic-units)")
        elif dahl < HALLUCINATION_THRESHOLD:                      # [FIX-THRESHOLD-2]
            flags.append(
                f"DAHL={dahl:.4f} < threshold={HALLUCINATION_THRESHOLD} "
                f"(v1 threshold was 0.30)"
            )

        if details.get("dim1_relevance", 1.0) < 0.35:          # [FIX-DIM1-5]
            flags.append(
                f"dim1_relevance={details['dim1_relevance']:.3f} < 0.35 "
                f"(v1 was < 0.30) — respons off-topic"
            )

        if r["filenames"] and details.get("dim2_grounding", 1.0) < 0.20:  # [FIX-GROUNDING-3]
            flags.append(
                f"dim2_grounding={details['dim2_grounding']:.3f} < 0.20 "
                f"(v1 was < 0.10) — respons tidak berdasar dokumen"
            )

        audited.append({
            **r,
            "hallucination_flag"  : len(flags) > 0,
            "hallucination_reason": "; ".join(flags) if flags else "OK",
        })
    return audited

# =========================================================
# CELL 13-18 (LOOPED) — EVALUASI TPS DIULANG UNTUK N_EPOCHS EPOCH
# =========================================================
# Setiap epoch = satu putaran penuh generate+skor untuk Baseline & DPO
# model (do_sample=True, temperature=TEMPERATURE tetap) sehingga hasil
# antar-epoch TIDAK identik secara token-per-token, tapi kalau metodenya
# valid, statistik agregatnya (avg_dahl, hallucination_rate, dst.) akan
# konsisten -> itulah yang dibuktikan lewat mean/std lintas-epoch di
# akhir (lihat CELL 18B).
#
# CHECKPOINT & RESUME (failsafe, robust, lintas akun Google/Colab):
#   - Tiap epoch punya file checkpoint sendiri:
#       baseline_results_checkpoint_epoch{N}.json
#       dpo_results_checkpoint_epoch{N}.json
#   - generate_responses_llm() sudah resume otomatis per-SAMPEL dari
#     checkpoint ini (skip 'kode' yang sudah ada), jadi walau sesi
#     Colab putus di tengah generate epoch ke-2 misalnya, tinggal
#     jalankan ulang notebook ini (dari akun Google manapun) selama
#     folder OUTPUT_DIR_EVAL yang di-mount adalah folder yang SAMA
#     (folder di-share ke akun lain -> tinggal 'Add shortcut to Drive'
#     supaya path-nya identik: My Drive/Thesis-Agung_Prawoto-56225110019).
#   - Kalau satu epoch SUDAH tercatat lengkap di epoch_summaries_progress.json,
#     epoch itu di-skip TOTAL (tidak load model sama sekali) di rerun berikutnya.
#   - Kalau epoch belum tercatat tapi checkpoint baseline/dpo-nya sudah
#     lengkap (>= jumlah dataset), model TIDAK di-load ulang (hemat
#     10-15 menit) — lihat pengecekan '_baseline_complete' / '_dpo_complete'.
#   - Semua checkpoint/JSON ditulis dengan atomic_json_save() (tulis ke
#     file .tmp lalu os.replace(), plus rotasi .bak) supaya file tidak
#     pernah setengah-tertulis/corrupt kalau runtime Colab mati mendadak
#     di tengah proses simpan.
# =========================================================

EPOCH_SUMMARIES = safe_json_load(
    os.path.join(OUTPUT_DIR_EVAL, "epoch_summaries_progress.json"), default=[]
)
_epochs_already_recorded = {s["epoch"] for s in EPOCH_SUMMARIES}
if _epochs_already_recorded:
    print(f"📂 Progres lintas-epoch ditemukan: epoch {sorted(_epochs_already_recorded)} "
          f"sudah tercatat sebelumnya di epoch_summaries_progress.json.")

for EPOCH in EPOCH_LIST:
    if EPOCH in _epochs_already_recorded:
        print(f"\n⏭️  Epoch {EPOCH}/{N_EPOCHS} sudah tercatat LENGKAP sebelumnya "
              f"(epoch_summaries_progress.json) — skip total, tidak load model.")
        continue

    print(f"\n{'#'*70}")
    print(f"# EPOCH {EPOCH}/{N_EPOCHS} — Evaluasi TPS (repetisi untuk cek konsistensi hasil)")
    print(f"{'#'*70}")

    # =========================================================
    # CELL 13 — LOAD BASELINE MODEL (Pre-DPO Evaluation)
    # =========================================================
    # Proteksi NameError: pulihkan dari JSON jika kernel restart
    if "all_contexts" not in globals() and "all_contexts" not in locals():
        precomputed_json_path = os.path.join(OUTPUT_DIR_EVAL, "precomputed_contexts.json")
        if os.path.exists(precomputed_json_path):
            print("📂 Memulihkan all_contexts dari backup JSON...")
            with open(precomputed_json_path, "r", encoding="utf-8") as f:
                all_contexts = json.load(f)
            print(f"✅ Data dipulihkan! Jumlah sampel: {len(all_contexts)}")
        else:
            raise NameError("❌ Gagal: jalankan Cell 7 terlebih dahulu!")

    baseline_ckpt = os.path.join(OUTPUT_DIR_EVAL, f"baseline_results_checkpoint_epoch{EPOCH}.json")

    # [FIX-SKIP-BASELINE] Cek apakah checkpoint sudah lengkap sebelum load model.
    # Load model 4-bit butuh ~10-15 menit + 8GB VRAM — tidak perlu kalau
    # semua 200 sampel sudah ada di checkpoint.
    _baseline_complete = False
    if os.path.exists(baseline_ckpt):
        try:
            _ckpt_data = safe_json_load(baseline_ckpt, default=[])
            if len(_ckpt_data) >= len(dataset):
                baseline_results = _ckpt_data
                # Re-apply hallucination_audit jika flag belum ada (checkpoint lama)
                if baseline_results and "hallucination_flag" not in baseline_results[0]:
                    print("🔄 Re-applying hallucination_audit pada checkpoint lama...")
                    baseline_results = hallucination_audit(baseline_results)
                _baseline_complete = True
                print(f"✅ Baseline checkpoint LENGKAP ({len(baseline_results)}/{len(dataset)} sampel) — skip load model & inference.")
            else:
                print(f"📂 Baseline checkpoint parsial ({len(_ckpt_data)}/{len(dataset)}) — lanjut dari sampel ke-{len(_ckpt_data)+1}.")
        except Exception as e:
            print(f"⚠️ Gagal baca baseline checkpoint ({e}) — mulai dari awal.")

    if not _baseline_complete:
        print(f"\nLoading BASELINE model: {LLM_MODEL_ID} ...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        llm_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

        baseline_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=llm_dtype,
            low_cpu_mem_usage=True,
            device_map="auto",
            token=HF_TOKEN,
            trust_remote_code=True,
        ).eval()
        baseline_model.config.use_cache = False

        baseline_tokenizer = AutoTokenizer.from_pretrained(
            LLM_MODEL_ID, token=HF_TOKEN, trust_remote_code=True
        )
        baseline_tokenizer.padding_side = "left"
        print("✅ Baseline model loaded.")
        print_mem_usage("setelah load baseline")

        baseline_results = generate_responses_llm(
            baseline_model,
            baseline_tokenizer,
            dataset,
            precomputed_contexts=all_contexts,
            label="Baseline (pre-DPO)",
            checkpoint_path=baseline_ckpt,
            save_every=5,
        )
        baseline_results = hallucination_audit(baseline_results)

        atomic_json_save(baseline_ckpt, baseline_results)
        print(f"✅ Baseline checkpoint disimpan (atomic): {baseline_ckpt}")

        del baseline_model
        clean_mem()
        print_mem_usage("setelah unload baseline")

    print("✅ Baseline evaluation selesai.")

    # =========================================================
    # CELL 14 — LOAD DPO MODEL (Post-DPO, dengan LoRA adapter)
    # =========================================================
    # json and os already imported in Cell 2
    if not os.path.exists(DPO_ADAPTER_PATH):
        raise FileNotFoundError(
            f"❌ DPO adapter tidak ditemukan: {DPO_ADAPTER_PATH}\n"
            "Pastikan Stage 3 sudah selesai dijalankan."
        )

    # Proteksi NameError
    if "all_contexts" not in globals() and "all_contexts" not in locals():
        precomputed_json_path = os.path.join(OUTPUT_DIR_EVAL, "precomputed_contexts.json")
        if os.path.exists(precomputed_json_path):
            with open(precomputed_json_path, "r", encoding="utf-8") as f:
                all_contexts = json.load(f)
        else:
            raise NameError("❌ Gagal: jalankan Cell 7 terlebih dahulu!")

    dpo_ckpt = os.path.join(OUTPUT_DIR_EVAL, f"dpo_results_checkpoint_epoch{EPOCH}.json")

    # [FIX-SKIP-DPO] Cek apakah checkpoint sudah lengkap sebelum load model.
    # Load DPO model (base + LoRA adapter) butuh ~10-15 menit + VRAM ekstra.
    _dpo_complete = False
    if os.path.exists(dpo_ckpt):
        try:
            _ckpt_data = safe_json_load(dpo_ckpt, default=[])
            if len(_ckpt_data) >= len(dataset):
                dpo_results = _ckpt_data
                if dpo_results and "hallucination_flag" not in dpo_results[0]:
                    print("🔄 Re-applying hallucination_audit pada DPO checkpoint lama...")
                    dpo_results = hallucination_audit(dpo_results)
                _dpo_complete = True
                print(f"✅ DPO checkpoint LENGKAP ({len(dpo_results)}/{len(dataset)} sampel) — skip load model & inference.")
            else:
                print(f"📂 DPO checkpoint parsial ({len(_ckpt_data)}/{len(dataset)}) — lanjut dari sampel ke-{len(_ckpt_data)+1}.")
        except Exception as e:
            print(f"⚠️ Gagal baca DPO checkpoint ({e}) — mulai dari awal.")

    if not _dpo_complete:
        print(f"\nLoading DPO model: {LLM_MODEL_ID} + adapter {DPO_ADAPTER_PATH} ...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        llm_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

        dpo_base = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=llm_dtype,
            low_cpu_mem_usage=True,
            device_map="auto",
            token=HF_TOKEN,
            trust_remote_code=True,
        )
        dpo_base.config.use_cache = False

        dpo_model = PeftModel.from_pretrained(dpo_base, DPO_ADAPTER_PATH)
        dpo_model.eval()

        dpo_tokenizer = AutoTokenizer.from_pretrained(
            LLM_MODEL_ID, token=HF_TOKEN, trust_remote_code=True
        )
        dpo_tokenizer.padding_side = "left"
        print("✅ DPO model loaded.")
        print_mem_usage("setelah load DPO")

        dpo_results = generate_responses_llm(
            dpo_model,
            dpo_tokenizer,
            dataset,
            precomputed_contexts=all_contexts,
            label="DPO Model (post-DPO)",
            checkpoint_path=dpo_ckpt,
            save_every=5,
        )
        dpo_results = hallucination_audit(dpo_results)

        atomic_json_save(dpo_ckpt, dpo_results)
        print(f"✅ DPO checkpoint disimpan (atomic): {dpo_ckpt}")

        del dpo_model, dpo_base
        clean_mem()
        print_mem_usage("setelah unload DPO")

    print("✅ DPO evaluation selesai.")

    # =========================================================
    # CELL 15 — HITUNG METRIK NLG
    # =========================================================
    print("\n" + "=" * 60)
    print("MENGHITUNG NLG METRICS")
    print("=" * 60)

    baseline_gen = [r["response"]  for r in baseline_results]
    baseline_ref = [r["reference"] for r in baseline_results]
    dpo_gen      = [r["response"]  for r in dpo_results]
    dpo_ref      = [r["reference"] for r in dpo_results]

    if HAS_REFERENCE:
        print("\n[1/2] Baseline NLG Metrics...")
        baseline_nlg = compute_nlg_metrics(baseline_gen, baseline_ref)

        print("\n[2/2] DPO Model NLG Metrics...")
        dpo_nlg = compute_nlg_metrics(dpo_gen, dpo_ref)

        if baseline_nlg and dpo_nlg:
            print("\n── NLG Results ──────────────────────────────────────")
            print(f"{'Metric':<15} {'Baseline':>10} {'DPO':>10} {'Delta':>10}")
            print("-" * 50)
            for metric in baseline_nlg:
                b     = baseline_nlg[metric]
                d     = dpo_nlg[metric]
                delta = d - b
                arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "─")
                print(f"{metric:<15} {b:>10.4f} {d:>10.4f} {arrow}{abs(delta):>8.4f}")
    else:
        print("\n⚠️  NLG metrics dilewati (tidak ada ground-truth reference).")
        baseline_nlg = {}
        dpo_nlg      = {}

    # =========================================================
    # CELL 16 — DAHL AGGREGATE + HALLUCINATION RATE
    # =========================================================
    print("\n" + "=" * 60)
    print("DAHL AGGREGATE + HALLUCINATION AUDIT (v2)")
    print(f"  Bobot DAHL : W1={DAHL_W1} (relevance), W2={DAHL_W2} (grounding), W3={DAHL_W3} (lexical)")
    print(f"  Threshold  : DAHL < {HALLUCINATION_THRESHOLD} | dim1 < 0.35 | dim2 < 0.20")
    print("=" * 60)


    def aggregate_dahl(results, label):
        """Hitung statistik agregat DAHL dan hallucination rate."""
        # [FIX] Filter None scores (from empty/unverifiable responses)
        scores    = [r["dahl_score"] for r in results if r.get("dahl_score") is not None]
        hall_cnt  = sum(1 for r in results if r["hallucination_flag"])
        dim1_list = [r["dahl_details"].get("dim1_relevance", 0) for r in results if r.get("dahl_details")]
        dim2_list = [r["dahl_details"].get("dim2_grounding", 0) for r in results if r.get("dahl_details")]
        dim3_list = [r["dahl_details"].get("dim3_lexical",   0) for r in results if r.get("dahl_details")]
        return {
            "model"              : label,
            "avg_dahl"           : round(float(np.mean(scores)), 4) if scores else 0.0,
            "min_dahl"           : round(float(np.min(scores)),  4) if scores else 0.0,
            "max_dahl"           : round(float(np.max(scores)),  4) if scores else 0.0,
            "std_dahl"           : round(float(np.std(scores)),  4) if scores else 0.0,
            "avg_dim1"           : round(float(np.mean(dim1_list)), 4) if dim1_list else 0.0,
            "avg_dim2"           : round(float(np.mean(dim2_list)), 4) if dim2_list else 0.0,
            "avg_dim3"           : round(float(np.mean(dim3_list)), 4) if dim3_list else 0.0,
            "hallucination_count": hall_cnt,
            "hallucination_rate" : round(hall_cnt / len(results), 4) if results else 0.0,
            "n_samples"          : len(results),
            "n_valid_scores"     : len(scores),
        }


    baseline_agg = aggregate_dahl(baseline_results, "Baseline (pre-DPO)")
    dpo_agg      = aggregate_dahl(dpo_results,      "DPO Model (post-DPO)")

    print(f"\n{'':32} {'Baseline':>12} {'DPO':>12}")
    print("-" * 58)
    for key in [
        "avg_dahl", "min_dahl", "max_dahl", "std_dahl",
        "avg_dim1", "avg_dim2", "avg_dim3",
        "hallucination_count", "hallucination_rate",
    ]:
        print(f"  {key:<30} {str(baseline_agg[key]):>12} {str(dpo_agg[key]):>12}")

    for model_name, results_list in [
        ("Baseline (pre-DPO)",   baseline_results),
        ("DPO Model (post-DPO)", dpo_results),
    ]:
        print(f"\n── Hallucination Detail: {model_name} ──")
        flagged = [r for r in results_list if r["hallucination_flag"]]
        ok_list = [r for r in results_list if not r["hallucination_flag"]]
        print(f"   Flagged: {len(flagged)} | OK: {len(ok_list)} | Total: {len(results_list)}")
        for r in results_list:
            icon = "🚨 HALUSINASI" if r["hallucination_flag"] else "✅ OK"
            d    = r["dahl_details"]
            print(
                f"  {icon} | [{r['kode']}] "
                f"DAHL={r['dahl_score']:.4f} "
                f"(d1={d.get('dim1_relevance',0):.3f} "
                f"d2={d.get('dim2_grounding',0):.3f} "
                f"d3={d.get('dim3_lexical',0):.3f}) "
                f"| {r['prompt'][:45]}..."
            )
            if r["hallucination_flag"]:
                print(f"          Reason: {r['hallucination_reason']}")

    # [FIX-JSON-ORDER] Simpan evaluation_detail.json di sini (akhir Cell 16),
    # SEBELUM Cell 17 plotting membacanya. Sebelumnya JSON baru disimpan di Cell 18
    # sehingga Cell 17 selalu masuk cabang "file not found" dan plot semua nol.
    _early_json_path = os.path.join(OUTPUT_DIR_EVAL, f"evaluation_detail_epoch{EPOCH}.json")
    _early_detail = {
        "dahl_version"    : "v2",
        "dahl_weights"    : {"W1": DAHL_W1, "W2": DAHL_W2, "W3": DAHL_W3},
        "hallucination_threshold": HALLUCINATION_THRESHOLD,
        "stop_words_count": len(STOP_WORDS_ID),
        "min_token_len"   : MIN_TOKEN_LEN,
        "baseline_results": baseline_results,
        "dpo_results"     : dpo_results,
        "baseline_nlg"    : baseline_nlg,
        "dpo_nlg"         : dpo_nlg,
        "baseline_dahl"   : baseline_agg,
        "dpo_dahl"        : dpo_agg,
        "dataset_size"    : len(dataset),
        "has_reference"   : HAS_REFERENCE,
    }
    atomic_json_save(_early_json_path, _early_detail)
    print(f"💾 evaluation_detail.json disimpan (sebelum plotting): {_early_json_path}")

    # =========================================================
    # CELL 17 — VISUALISASI (5 PANEL)
    # Panel baru dibanding v1: panel 5 = per-sampel dim2_grounding
    # untuk visualisasi grounding quality yang lebih eksplisit.
    # =========================================================
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec

    # Re-define constants for independent execution
    OUTPUT_DIR_EVAL   = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/OUTPUT_EVAL"
    HALLUCINATION_THRESHOLD = 0.50

    # Load results from evaluation_detail.json saved at end of Cell 16.
    # [FIX-JSON-ORDER] JSON is now always written before this cell runs.
    json_path = os.path.join(OUTPUT_DIR_EVAL, f"evaluation_detail_epoch{EPOCH}.json")

    # [FIX-KEYERROR] Full set of required keys used as safe defaults.
    # Merging loaded dict with this ensures old JSON files (missing 'avg_dahl' etc.)
    # never cause KeyError downstream in the summary_rows loop.
    _EMPTY_AGG = {
        "avg_dahl": 0.0, "min_dahl": 0.0, "max_dahl": 0.0, "std_dahl": 0.0,
        "avg_dim1": 0.0, "avg_dim2": 0.0, "avg_dim3": 0.0,
        "hallucination_count": 0, "hallucination_rate": 0.0,
        "n_samples": 0, "n_valid_scores": 0,
    }

    if os.path.exists(json_path):
        print(f"Loading evaluation details from {json_path}...")
        with open(json_path, "r", encoding="utf-8") as f:
            evaluation_detail = json.load(f)
        baseline_results = evaluation_detail.get("baseline_results", [])
        dpo_results      = evaluation_detail.get("dpo_results", [])
        baseline_nlg     = evaluation_detail.get("baseline_nlg", {})
        dpo_nlg          = evaluation_detail.get("dpo_nlg", {})
        # Merge with defaults so any key absent in old JSON is safely zero-filled
        baseline_agg = {**_EMPTY_AGG, **evaluation_detail.get("baseline_dahl", {})}
        dpo_agg      = {**_EMPTY_AGG, **evaluation_detail.get("dpo_dahl",      {})}
        HAS_REFERENCE = evaluation_detail.get("has_reference", False)
    else:
        print("Warning: evaluation_detail.json not found. Plotting will be limited.")
        baseline_results = []
        dpo_results      = []
        baseline_nlg     = {}
        dpo_nlg          = {}
        baseline_agg     = _EMPTY_AGG.copy()
        dpo_agg          = _EMPTY_AGG.copy()
        HAS_REFERENCE    = False


    fig = plt.figure(figsize=(28, 10))
    gs  = gridspec.GridSpec(1, 5, figure=fig, wspace=0.40)

    fig.suptitle(
        "VLM Alignment Evaluation: Baseline vs DPO Model\n"
        "TPS Evaluation | Threshold=0.50",
        fontsize=13, fontweight="bold", y=1.01,
    )

    C_BASE = "#4C9BE8"
    C_DPO  = "#E8834C"
    C_RED  = "#E84C4C"
    C_GRN  = "#4CB87A"

    # ── Panel 1: NLG Metrics ───────────────────────────────────
    ax0 = fig.add_subplot(gs[0])
    if baseline_nlg and dpo_nlg and HAS_REFERENCE:
        names = list(baseline_nlg.keys())
        x, w  = np.arange(len(names)), 0.35
        ax0.bar(x - w/2, [baseline_nlg[m] for m in names], w, label="Baseline",  color=C_BASE, alpha=0.85)
        ax0.bar(x + w/2, [dpo_nlg[m]      for m in names], w, label="DPO Model", color=C_DPO,  alpha=0.85)
        ax0.set_title("NLG Metrics")
        ax0.set_xticks(x)
        ax0.set_xticklabels(names, rotation=25, ha="right", fontsize=8)
        ax0.set_ylim(0, 1); ax0.set_ylabel("Score")
        ax0.legend(fontsize=8); ax0.grid(axis="y", linestyle="--", alpha=0.6)
    else:
        ax0.text(0.5, 0.5, "NLG metrics\nnot available",
                 ha="center", va="center", transform=ax0.transAxes, color="gray", style="italic")
        ax0.set_title("NLG Metrics")

    # ── Panel 2: DAHL Score per sampel ─────────────────────────
    ax1 = fig.add_subplot(gs[1])
    if baseline_results and dpo_results:
        idx = np.arange(len(baseline_results))
        ax1.plot(idx, [r["dahl_score"] for r in baseline_results],
                 "o--", color=C_BASE, label="Baseline",  linewidth=1.2, markersize=4)
        ax1.plot(idx, [r["dahl_score"] for r in dpo_results],
                 "s-",  color=C_DPO,  label="DPO Model", linewidth=1.2, markersize=4)
        ax1.axhline(y=HALLUCINATION_THRESHOLD, color=C_RED, linestyle=":",
                    linewidth=1.3, label=f"Threshold ({HALLUCINATION_THRESHOLD})")
        ax1.set_title("TPS Score per Sample")
        ax1.set_xlabel("Sample"); ax1.set_ylabel("TPS Score")
        ax1.set_ylim(0, 1); ax1.legend(fontsize=7)
        ax1.grid(True, linestyle="--", alpha=0.5)
    else:
        ax1.text(0.5, 0.5, "TPS data\nnot available",
                 ha="center", va="center", transform=ax1.transAxes, color="gray", style="italic")
        ax1.set_title("TPS Score per Sample")

    # ── Panel 3: Hallucination Rate ─────────────────────────────
    ax2 = fig.add_subplot(gs[2])
    if baseline_agg and dpo_agg:
        h_rates = [baseline_agg["hallucination_rate"], dpo_agg["hallucination_rate"]]
        bars    = ax2.bar(["Baseline", "DPO Model"], h_rates, color=[C_BASE, C_DPO], alpha=0.85, width=0.4)
        for bar, rate in zip(bars, h_rates):
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                     f"{rate:.2%}", ha="center", va="bottom", fontweight="bold", fontsize=10)
        ax2.set_title("Hallucination Rate (threshold=0.50)")
        ax2.set_ylabel("Rate"); ax2.set_ylim(0, 1)
        ax2.grid(axis="y", linestyle="--", alpha=0.6)
    else:
        ax2.text(0.5, 0.5, "Hallucination Rate\ndata not available",
                 ha="center", va="center", transform=ax2.transAxes, color="gray", style="italic")
        ax2.set_title("Hallucination Rate (threshold=0.50)")

    # ── Panel 4: DAHL Dimension Breakdown ───────────────────────
    ax3 = fig.add_subplot(gs[3])
    if baseline_agg and dpo_agg:
        dim_labels = ["Dim1\nRelevance\n(w=0.40)", "Dim2\nGrounding\n(w=0.40)", "Dim3\nLexical\n(w=0.20)"]
        base_dims  = [baseline_agg["avg_dim1"], baseline_agg["avg_dim2"], baseline_agg["avg_dim3"]]
        dpo_dims   = [dpo_agg["avg_dim1"],      dpo_agg["avg_dim2"],      dpo_agg["avg_dim3"]]
        x_dim, w_dim = np.arange(len(dim_labels)), 0.30
        b1 = ax3.bar(x_dim - w_dim/2, base_dims, w_dim, label="Baseline",  color=C_BASE, alpha=0.85)
        b2 = ax3.bar(x_dim + w_dim/2, dpo_dims,  w_dim, label="DPO Model", color=C_DPO,  alpha=0.85)
        for bar in list(b1) + list(b2):
            ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                     f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7)
        ax3.set_title("TPS Dimension Breakdown\n(avg per model)")
        ax3.set_xticks(x_dim); ax3.set_xticklabels(dim_labels, fontsize=8)
        ax3.set_ylim(0, 1); ax3.set_ylabel("Avg Score")
        ax3.legend(fontsize=8); ax3.grid(axis="y", linestyle="--", alpha=0.6)
    else:
        ax3.text(0.5, 0.5, "TPS Dimension\ndata not available",
                 ha="center", va="center", transform=ax3.transAxes, color="gray", style="italic")
        ax3.set_title("TPS Dimension Breakdown\n(avg per model)")

    # ── Panel 5 (BARU): dim2_grounding per sampel ───────────────
    # Menampilkan grounding quality tiap sampel secara eksplisit.
    # Sampel di bawah threshold (0.20) = kandidat halusinasi.
    ax4 = fig.add_subplot(gs[4])
    if baseline_results and dpo_results:
        idx = np.arange(len(baseline_results))
        ax4.plot(idx, [r["dahl_details"].get("dim2_grounding", 0) for r in baseline_results],
                 "o--", color=C_BASE, label="Baseline",  linewidth=1.2, markersize=4)
        ax4.plot(idx, [r["dahl_details"].get("dim2_grounding", 0) for r in dpo_results],
                 "s-",  color=C_DPO,  label="DPO Model", linewidth=1.2, markersize=4)
        ax4.axhline(y=0.20, color=C_RED, linestyle=":",
                    linewidth=1.3, label="Grounding thr. (0.20)")
        ax4.set_title("Dim2 Grounding per Sample\n(direct correlation with document)")
        ax4.set_xlabel("Sample"); ax4.set_ylabel("dim2_grounding")
        ax4.set_ylim(0, 1); ax4.legend(fontsize=7)
        ax4.grid(True, linestyle="--", alpha=0.5)
    else:
        ax4.text(0.5, 0.5, "Grounding per Sample\ndata not available",
                 ha="center", va="center", transform=ax4.transAxes, color="gray", style="italic")
        ax4.set_title("Dim2 Grounding per Sample\n(direct correlation with document)")

    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR_EVAL, f"evaluation_comparison_epoch{EPOCH}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Plot disimpan: {plot_path}")

    # =========================================================
    # CELL 18 — SIMPAN HASIL EVALUASI (CSV + JSON)
    # Ditambahkan kolom 'dahl_version' untuk membedakan output v1 dan v2.
    # =========================================================

    # ── Summary CSV ───────────────────────────────────────────
    summary_rows = []

    # Metadata versi — memudahkan tracking saat membandingkan v1 vs v2
    for model_label, agg in [("Baseline", baseline_agg), ("DPO", dpo_agg)]:
        summary_rows.append({
            "metric"         : "dahl_version",
            "baseline"       : "v2 (W1=0.40,W2=0.40,W3=0.20,thr=0.50)" if model_label == "Baseline" else "",
            "dpo_model"      : "v2 (W1=0.40,W2=0.40,W3=0.20,thr=0.50)" if model_label == "DPO" else "",
            "delta"          : "",
            "improvement_pct": "",
            "category"       : "Meta",
        })
        break  # cukup satu baris metadata

    if baseline_nlg and dpo_nlg:
        for metric in baseline_nlg:
            b = baseline_nlg[metric]
            d = dpo_nlg[metric]
            summary_rows.append({
                "metric"         : metric,
                "baseline"       : b,
                "dpo_model"      : d,
                "delta"          : round(d - b, 4),
                "improvement_pct": round((d - b) / max(b, 1e-9) * 100, 2),
                "category"       : "NLG Metric",
            })

    for key in [
        "avg_dahl", "min_dahl", "max_dahl", "std_dahl",
        "avg_dim1", "avg_dim2", "avg_dim3",
        "hallucination_count", "hallucination_rate",
    ]:
        # [FIX-KEYERROR] Use .get() with 0 fallback — guards against old JSON files
        # that were saved before these keys existed in aggregate_dahl() output.
        b = baseline_agg.get(key, 0.0)
        d = dpo_agg.get(key, 0.0)
        summary_rows.append({
            "metric"         : key,
            "baseline"       : b,
            "dpo_model"      : d,
            "delta"          : round(float(d) - float(b), 4),
            "improvement_pct": round((float(d) - float(b)) / max(abs(float(b)), 1e-9) * 100, 2),
            "category"       : "DAHL" if ("dahl" in key or "dim" in key) else "Hallucination",
        })

    summary_df = pd.DataFrame(summary_rows)
    csv_path   = os.path.join(OUTPUT_DIR_EVAL, f"evaluation_summary_epoch{EPOCH}.csv")
    summary_df.to_csv(csv_path, index=False)
    print(f"✅ Summary CSV: {csv_path}")

    # ── Per-sample Detail CSV ─────────────────────────────────
    def flatten_results(results):
        rows = []
        for r in results:
            rows.append({
                "kode"                : r.get("kode", ""),
                "topik"               : r.get("topik", ""),
                "model_label"         : r["model_label"],
                "prompt"              : r["prompt"],
                "reference"           : r["reference"],
                "response"            : r["response"],
                "filenames"           : "|".join(r["filenames"]),
                "dahl_score"          : r["dahl_score"],
                "dahl_version"        : "v2",
                "dahl_weights"        : f"W1={DAHL_W1},W2={DAHL_W2},W3={DAHL_W3}",
                "dim1_relevance"      : r["dahl_details"].get("dim1_relevance", ""),
                "dim2_grounding"      : r["dahl_details"].get("dim2_grounding", ""),
                "dim3_lexical"        : r["dahl_details"].get("dim3_lexical", ""),
                "hallucination_flag"  : r["hallucination_flag"],
                "hallucination_reason": r["hallucination_reason"],
            })
        return rows

    detail_df = pd.DataFrame(
        flatten_results(baseline_results) + flatten_results(dpo_results)
    )
    detail_csv_path = os.path.join(OUTPUT_DIR_EVAL, f"evaluation_per_sample_epoch{EPOCH}.csv")
    detail_df.to_csv(detail_csv_path, index=False)
    print(f"✅ Per-sample CSV: {detail_csv_path}")

    # ── Detail JSON ───────────────────────────────────────────
    detail_output = {
        "dahl_version"    : "v2",
        "dahl_weights"    : {"W1": DAHL_W1, "W2": DAHL_W2, "W3": DAHL_W3},
        "hallucination_threshold": HALLUCINATION_THRESHOLD,
        "stop_words_count": len(STOP_WORDS_ID),
        "min_token_len"   : MIN_TOKEN_LEN,
        "baseline_results": baseline_results,
        "dpo_results"     : dpo_results,
        "baseline_nlg"    : baseline_nlg,
        "dpo_nlg"         : dpo_nlg,
        "baseline_dahl"   : baseline_agg,
        "dpo_dahl"        : dpo_agg,
        "dataset_size"    : len(dataset),
        "has_reference"   : HAS_REFERENCE,
    }
    json_path = os.path.join(OUTPUT_DIR_EVAL, f"evaluation_detail_epoch{EPOCH}.json")
    atomic_json_save(json_path, detail_output)
    print(f"✅ Detail JSON : {json_path}")

    # ── Tampilkan ringkasan ────────────────────────────────────
    print("\n" + "=" * 60)
    print("RINGKASAN EVALUASI AKHIR (Stage 4)")
    print("=" * 60)
    print(summary_df.to_string(index=False))

    print("\n✅ Stage 4 selesai.")
    print(f"   Output tersimpan di: {OUTPUT_DIR_EVAL}")
    print(f"   File output (epoch {EPOCH}):")
    print(f"     • evaluation_summary_epoch{EPOCH}.csv      — ringkasan semua metrik")
    print(f"     • evaluation_per_sample_epoch{EPOCH}.csv   — detail per baris (baseline + DPO)")
    print(f"     • evaluation_detail_epoch{EPOCH}.json      — full JSON dengan semua hasil")
    print(f"     • evaluation_comparison_epoch{EPOCH}.png   — 5-panel visualisasi")
    print("     • precomputed_contexts.json      — backup contexts retrieval (shared, sekali retrieval)")
    print(f"     • baseline_results_checkpoint_epoch{EPOCH}.json")
    print(f"     • dpo_results_checkpoint_epoch{EPOCH}.json")

    # --- Simpan ringkasan epoch ini ke daftar EPOCH_SUMMARIES (untuk analisis lintas-epoch) ---
    EPOCH_SUMMARIES.append({
        "epoch"        : EPOCH,
        "baseline_agg" : baseline_agg,
        "dpo_agg"      : dpo_agg,
        "baseline_nlg" : baseline_nlg,
        "dpo_nlg"      : dpo_nlg,
    })
    atomic_json_save(os.path.join(OUTPUT_DIR_EVAL, "epoch_summaries_progress.json"), EPOCH_SUMMARIES)
    print(f"\n🏁 EPOCH {EPOCH}/{N_EPOCHS} SELESAI dan tercatat di epoch_summaries_progress.json.\n")

# =========================================================
# CELL 18B — RINGKASAN LINTAS EPOCH (mean ± std, N_EPOCHS repetisi)
# =========================================================
# Tujuan: membuktikan hasil evaluasi TPS BUKAN kebetulan satu kali run.
# Karena generate_responses_llm() memakai do_sample=True (temperature
# tetap = TEMPERATURE), tiap epoch menghasilkan teks yang bisa berbeda
# kata-per-kata, tapi kalau metode & modelnya valid, statistik agregatnya
# (avg_dahl, hallucination_rate, metrik NLG) seharusnya STABIL antar
# epoch — itulah yang direkap di sini sebagai mean ± std.
# =========================================================
print("\n" + "=" * 70)
print(f"RINGKASAN LINTAS {N_EPOCHS} EPOCH (uji konsistensi hasil TPS)")
print("=" * 70)

if len(EPOCH_SUMMARIES) < N_EPOCHS:
    print(
        f"⚠️  Baru {len(EPOCH_SUMMARIES)}/{N_EPOCHS} epoch yang tercatat lengkap "
        f"di epoch_summaries_progress.json.\n"
        f"    Jalankan ulang notebook ini (boleh dari akun Google/Colab lain, "
        f"selama folder OUTPUT_DIR_EVAL yang sama sudah di-mount) untuk "
        f"melanjutkan epoch yang tersisa sebelum melihat ringkasan final."
    )

_rows = []
for model_key, agg_key, nlg_key, label in [
    ("baseline", "baseline_agg", "baseline_nlg", "Baseline (pre-DPO)"),
    ("dpo",      "dpo_agg",      "dpo_nlg",      "DPO Model (post-DPO)"),
]:
    dahl_vals = [s[agg_key].get("avg_dahl", np.nan) for s in EPOCH_SUMMARIES]
    hall_vals = [s[agg_key].get("hallucination_rate", np.nan) for s in EPOCH_SUMMARIES]
    dim1_vals = [s[agg_key].get("avg_dim1", np.nan) for s in EPOCH_SUMMARIES]
    dim2_vals = [s[agg_key].get("avg_dim2", np.nan) for s in EPOCH_SUMMARIES]
    dim3_vals = [s[agg_key].get("avg_dim3", np.nan) for s in EPOCH_SUMMARIES]

    row = {
        "model"                  : label,
        "n_epochs_completed"     : len(EPOCH_SUMMARIES),
        "avg_dahl_mean"          : round(float(np.nanmean(dahl_vals)), 4) if dahl_vals else None,
        "avg_dahl_std"           : round(float(np.nanstd(dahl_vals)), 4) if dahl_vals else None,
        "hallucination_rate_mean": round(float(np.nanmean(hall_vals)), 4) if hall_vals else None,
        "hallucination_rate_std" : round(float(np.nanstd(hall_vals)), 4) if hall_vals else None,
        "avg_dim1_mean"          : round(float(np.nanmean(dim1_vals)), 4) if dim1_vals else None,
        "avg_dim2_mean"          : round(float(np.nanmean(dim2_vals)), 4) if dim2_vals else None,
        "avg_dim3_mean"          : round(float(np.nanmean(dim3_vals)), 4) if dim3_vals else None,
    }

    # NLG metrics (kalau ground-truth tersedia)
    if EPOCH_SUMMARIES and EPOCH_SUMMARIES[0].get(nlg_key):
        for metric_name in EPOCH_SUMMARIES[0][nlg_key]:
            vals = [s[nlg_key].get(metric_name, np.nan) for s in EPOCH_SUMMARIES if s.get(nlg_key)]
            row[f"{metric_name}_mean"] = round(float(np.nanmean(vals)), 4) if vals else None
            row[f"{metric_name}_std"]  = round(float(np.nanstd(vals)), 4) if vals else None

    # Detail per-epoch (biar bisa lihat sebaran mentahnya, bukan cuma mean±std)
    row["avg_dahl_per_epoch"] = [round(v, 4) if v == v else None for v in dahl_vals]
    row["hallucination_rate_per_epoch"] = [round(v, 4) if v == v else None for v in hall_vals]

    _rows.append(row)

cross_epoch_df = pd.DataFrame(_rows)
cross_epoch_csv_path = os.path.join(OUTPUT_DIR_EVAL, "evaluation_summary_across_epochs.csv")
cross_epoch_df.to_csv(cross_epoch_csv_path, index=False)
atomic_json_save(
    os.path.join(OUTPUT_DIR_EVAL, "evaluation_summary_across_epochs.json"),
    {"n_epochs_target": N_EPOCHS, "epoch_summaries": EPOCH_SUMMARIES, "cross_epoch": _rows},
)

print(f"\n✅ Ringkasan lintas-epoch disimpan: {cross_epoch_csv_path}")
for row in _rows:
    print(f"\n── {row['model']} (n_epoch={row['n_epochs_completed']}/{N_EPOCHS}) ──")
    print(f"   avg_dahl            : {row['avg_dahl_mean']} ± {row['avg_dahl_std']}  "
          f"(per-epoch: {row['avg_dahl_per_epoch']})")
    print(f"   hallucination_rate  : {row['hallucination_rate_mean']} ± {row['hallucination_rate_std']}  "
          f"(per-epoch: {row['hallucination_rate_per_epoch']})")

# ── Plot konsistensi avg_dahl per-epoch (Baseline vs DPO) ──────────────
if len(EPOCH_SUMMARIES) >= 1:
    try:
        import matplotlib.pyplot as _plt
        _epochs_x   = [s["epoch"] for s in EPOCH_SUMMARIES]
        _base_dahl  = [s["baseline_agg"].get("avg_dahl", np.nan) for s in EPOCH_SUMMARIES]
        _dpo_dahl   = [s["dpo_agg"].get("avg_dahl", np.nan) for s in EPOCH_SUMMARIES]

        _fig, _ax = _plt.subplots(figsize=(7, 4.5))
        _ax.plot(_epochs_x, _base_dahl, "o--", color="#4C9BE8", label="Baseline", markersize=8)
        _ax.plot(_epochs_x, _dpo_dahl,  "s-",  color="#E8834C", label="DPO Model", markersize=8)
        _ax.set_xticks(_epochs_x)
        _ax.set_xlabel("Epoch (repetisi evaluasi)")
        _ax.set_ylabel("Avg TPS (DAHL proxy) Score")
        _ax.set_title(f"Konsistensi Avg TPS Score across {N_EPOCHS} Epoch")
        _ax.set_ylim(0, 1)
        _ax.legend()
        _ax.grid(True, linestyle="--", alpha=0.5)
        _plt.tight_layout()
        _consistency_plot_path = os.path.join(OUTPUT_DIR_EVAL, "evaluation_consistency_across_epochs.png")
        _plt.savefig(_consistency_plot_path, dpi=150, bbox_inches="tight")
        _plt.show()
        print(f"✅ Plot konsistensi lintas-epoch disimpan: {_consistency_plot_path}")
    except Exception as e:
        print(f"⚠️ Gagal membuat plot konsistensi lintas-epoch: {e}")

print("\n" + "=" * 70)
print("CATATAN untuk CELL 19 dst. (validasi independen / statistical testing):")
print(f"  baseline_results, dpo_results, baseline_agg, dpo_agg, baseline_nlg,")
print(f"  dpo_nlg di bawah ini masih berisi data dari EPOCH TERAKHIR ({N_EPOCHS}) saja.")
print(f"  evaluation_per_sample.csv (dipakai CELL 21) akan dibuat ulang dari data")
print(f"  epoch terakhir tsb agar cell-cell selanjutnya tetap berjalan tanpa")
print(f"  perlu diubah. Kalau butuh statistik lintas-epoch, pakai")
print(f"  evaluation_summary_across_epochs.csv/.json yang baru saja disimpan.")
print("=" * 70)

# Cell 19-21 (kode asli) membaca "evaluation_per_sample.csv" (tanpa suffix
# epoch) -- regenerasi file itu di sini dari hasil epoch TERAKHIR supaya
# cell-cell tersebut tetap jalan tanpa modifikasi lebih lanjut.
# [FAILSAFE] Jika semua epoch sudah tercatat (loop di atas hanya 'skip'),
# variabel hasil epoch terakhir tidak ada di memori -> pulihkan dari checkpoint.
if "flatten_results" not in globals():
    def flatten_results(results):
        rows = []
        for r in results:
            rows.append({
                "kode"                : r.get("kode", ""),
                "topik"               : r.get("topik", ""),
                "model_label"         : r["model_label"],
                "prompt"              : r["prompt"],
                "reference"           : r["reference"],
                "response"            : r["response"],
                "filenames"           : "|".join(r["filenames"]),
                "dahl_score"          : r["dahl_score"],
                "dahl_version"        : "v2",
                "dahl_weights"        : f"W1={DAHL_W1},W2={DAHL_W2},W3={DAHL_W3}",
                "dim1_relevance"      : r["dahl_details"].get("dim1_relevance", ""),
                "dim2_grounding"      : r["dahl_details"].get("dim2_grounding", ""),
                "dim3_lexical"        : r["dahl_details"].get("dim3_lexical", ""),
                "hallucination_flag"  : r["hallucination_flag"],
                "hallucination_reason": r["hallucination_reason"],
            })
        return rows

for _name, _tmpl in (("baseline_results", "baseline_results_checkpoint_epoch{}.json"),
                     ("dpo_results",      "dpo_results_checkpoint_epoch{}.json")):
    if _name not in globals():
        _p = os.path.join(OUTPUT_DIR_EVAL, _tmpl.format(N_EPOCHS))
        _data = safe_json_load(_p, default=[])
        if not _data:
            raise RuntimeError(f"❌ {_name} tidak ada di memori dan checkpoint kosong: {_p}")
        globals()[_name] = _data
        print(f"🔄 {_name} dipulihkan dari checkpoint epoch {N_EPOCHS} ({len(_data)} baris).")

# [FAILSAFE] baseline_agg/dpo_agg/baseline_nlg/dpo_nlg juga cuma ada di
# memori kalau loop epoch benar-benar jalan. Kalau semua epoch di-skip
# (sudah lengkap sebelumnya), pulihkan dari EPOCH_SUMMARIES (epoch terakhir
# yang tercatat), yang sudah memuat persis dict-dict ini.
if "baseline_agg" not in globals() or "dpo_agg" not in globals():
    if EPOCH_SUMMARIES:
        _last_summary = max(EPOCH_SUMMARIES, key=lambda s_: s_["epoch"])
        baseline_agg = _last_summary["baseline_agg"]
        dpo_agg      = _last_summary["dpo_agg"]
        baseline_nlg = _last_summary.get("baseline_nlg", {})
        dpo_nlg      = _last_summary.get("dpo_nlg", {})
        print(f"🔄 baseline_agg/dpo_agg/baseline_nlg/dpo_nlg dipulihkan dari "
              f"EPOCH_SUMMARIES epoch {_last_summary['epoch']}.")
    else:
        raise RuntimeError("❌ baseline_agg/dpo_agg tidak ada di memori dan "
                            "EPOCH_SUMMARIES kosong — jalankan loop epoch dulu.")


_compat_detail_df = pd.DataFrame(
    flatten_results(baseline_results) + flatten_results(dpo_results)
)
_compat_csv_path = os.path.join(OUTPUT_DIR_EVAL, "evaluation_per_sample.csv")
_compat_detail_df.to_csv(_compat_csv_path, index=False)
print(f"✅ (compat) evaluation_per_sample.csv diperbarui dari epoch {N_EPOCHS} untuk CELL 19-21: {_compat_csv_path}")

# =========================================================
# CELL 19 — VALIDASI INDEPENDEN: DAHL Score (Baseline vs DPO)
# (opsional: ablation temperature, replikasi Figure 5 paper Seo et al., 2024)
# =========================================================
#
# TUJUAN UTAMA cell ini (untuk menjawab revisi reviewer "Include
# independent hallucination validation not based on the same TPS metric
# used in training"): hitung DAHL Score & hallucination rate (atomic-unit,
# CELL 9B, sesuai paper Seo et al. 2024) untuk Baseline DAN DPO Model pada
# temperature default (sama dengan eval utama, TEMPERATURE=0.4) — sehingga
# klaim "DPO mengurangi halusinasi" diuji ulang dengan metodologi yang
# independen dari TPS (cosine similarity) dan dari training itu sendiri.
#
# Default ABLATION_TEMPERATURES cuma 1 nilai (TEMPERATURE) → cell ini
# berjalan 2x (Baseline, DPO Model), BUKAN grid 0.1–1.0 — itu studi
# tambahan yang OPSIONAL (efek temperature terhadap halusinasi, mirip
# Section 5.2 paper) dan TIDAK diperlukan untuk menjawab revisi reviewer.
# Kalau mau studi temperature juga, ganti baris ABLATION_TEMPERATURES di
# bawah jadi mis. [0.1, 0.3, 0.5, 0.7, 0.9, 1.0] — tapi sadari itu
# menjalankan generate+split+check 6x lebih banyak.
#
# Checker dicek terhadap teks konteks yang di-retrieve ColPali (gratis,
# pakai model lokal yang sudah ada) — bukan API eksternal.
#
# FIXED CHECKER: Splitter & Checker dijalankan oleh 1 model TETAP
# (baseline, tanpa adapter DPO) — sama untuk menilai Baseline maupun
# DPO. Tujuannya supaya DPO Model tidak menilai jawabannya sendiri.
# Konsekuensi: selama evaluasi DPO, ADA 2 model 7B (4-bit) di VRAM
# sekaligus (DPO sebagai generator + baseline sebagai checker) — kalau
# OOM di GPU kecil, turunkan ABLATION_SAMPLE_FRAC atau MAX_NEW_TOKENS.
# =========================================================

ABLATION_TEMPERATURES = [TEMPERATURE]   # default: cuma temperature eval utama (Baseline vs DPO)
# ABLATION_TEMPERATURES = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]   # <- uncomment untuk studi temperature (opsional)
ABLATION_SEED         = 42

# Dengan ABLATION_TEMPERATURES default cuma 1 nilai, cell ini cuma
# jalan 2x (Baseline, DPO) — bukan 12x seperti versi ablation suhu
# penuh. Gratis (model lokal) → pakai FULL dataset (100%) by default.
# Dataset eval biasanya kecil (±27-50 baris) — sample 15% dari itu cuma
# ~4-8 baris, terlalu sedikit untuk klaim yang meyakinkan.
# Kalau nanti ABLATION_TEMPERATURES diisi multi-value (studi temperature),
# turunkan fraksinya manual supaya total waktu run tetap wajar.
ABLATION_SAMPLE_FRAC = 1.0
print(f"DAHL Score independent validation — sample_frac={ABLATION_SAMPLE_FRAC:.0%}, "
      f"n_temperature={len(ABLATION_TEMPERATURES)}")

# Proteksi NameError: pulihkan all_contexts dari backup JSON jika kernel
# restart (pola sama seperti CELL 13), supaya tidak perlu ulang CELL 7
# (ColPali retrieval) yang paling mahal. 'dataset' TIDAK ada fallback
# otomatis karena murah — cukup jalankan ulang CELL 5 (load CSV) saja.
if "all_contexts" not in globals() and "all_contexts" not in locals():
    precomputed_json_path = os.path.join(OUTPUT_DIR_EVAL, "precomputed_contexts.json")
    if os.path.exists(precomputed_json_path):
        print("📂 Memulihkan all_contexts dari backup JSON...")
        with open(precomputed_json_path, "r", encoding="utf-8") as f:
            all_contexts = json.load(f)
        print(f"✅ Data dipulihkan! Jumlah sampel: {len(all_contexts)}")
    else:
        raise NameError("❌ all_contexts tidak ditemukan — jalankan CELL 5–7 terlebih dahulu!")

if "dataset" not in globals() and "dataset" not in locals():
    raise NameError("❌ dataset tidak ditemukan — jalankan CELL 5 terlebih dahulu (murah, tidak perlu ColPali)!")


def stratified_sample_indices(dataset, frac, category_col="topik", seed=ABLATION_SEED):
    """Sampling proporsional per kategori, meniru pendekatan paper
    ('maintaining the same categorical distribution as the full dataset')."""
    df_idx = pd.DataFrame({"idx": range(len(dataset)), "cat": dataset[category_col]})
    sampled = (
        df_idx.groupby("cat", group_keys=False)
        .apply(lambda g: g.sample(max(1, int(round(len(g) * frac))), random_state=seed))
    )
    return sorted(sampled["idx"].tolist())


_ablation_idx     = stratified_sample_indices(dataset, ABLATION_SAMPLE_FRAC)
ablation_dataset  = dataset.select(_ablation_idx)
ablation_contexts = [all_contexts[i] for i in _ablation_idx]
print(
    f"\nAblation sample: {len(ablation_dataset)}/{len(dataset)} baris "
    f"({len(ablation_dataset)/len(dataset):.1%}), stratified by 'topik'."
)


def load_ablation_model(use_dpo: bool):
    """Reload baseline atau DPO model untuk ablation (model utama
    sebelumnya sudah di-unload setelah evaluasi di CELL 13/14)."""
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    base = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        quantization_config=bnb_cfg,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        device_map="auto",
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    base.config.use_cache = False
    model = PeftModel.from_pretrained(base, DPO_ADAPTER_PATH) if use_dpo else base
    model.eval()
    tok = AutoTokenizer.from_pretrained(LLM_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
    tok.padding_side = "left"
    return model, tok


ablation_records      = []   # baris ringkas: model_label, temperature, avg_dahl, ...
ablation_raw_results  = {}   # hasil mentah per (model_label, temperature), untuk audit manual


def _ablation_ckpt_path(model_label: str, temp: float) -> str:
    safe_label = model_label.replace(" ", "_")
    return os.path.join(OUTPUT_DIR_EVAL, f"ablation_ckpt_{safe_label}_T{temp}.json")


def _ckpt_is_complete(path: str, n_expected: int) -> bool:
    if not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            return len(json.load(f)) >= n_expected
    except Exception:
        return False


# --- Fixed Checker: 1 model yang sama dipakai untuk menilai Baseline
# MAUPUN DPO, supaya model yang dievaluasi tidak menilai jawabannya
# sendiri. Dipilih model baseline (tanpa adapter DPO) — netral, tidak
# condong ke salah satu sisi yang dibandingkan.
print(f"\n{'='*60}\nMemuat Fixed Checker (baseline, tanpa adapter DPO)\n{'='*60}")
fixed_checker_model, fixed_checker_tokenizer = load_ablation_model(use_dpo=False)

for use_dpo, model_label in [(False, "Baseline"), (True, "DPO Model")]:
    temp_ckpt_paths = {temp: _ablation_ckpt_path(model_label, temp) for temp in ABLATION_TEMPERATURES}

    # Kalau SEMUA temperature utk model ini sudah lengkap di checkpoint
    # (dari run sebelumnya yang disconnect setelah model ini selesai),
    # skip reload model 7B sama sekali — hemat 1-3 menit reload.
    all_temps_done = all(
        _ckpt_is_complete(p, len(ablation_dataset)) for p in temp_ckpt_paths.values()
    )

    if all_temps_done:
        print(f"\n✅ {model_label}: semua temperature sudah lengkap di checkpoint — skip reload model.")
        abl_model, abl_tokenizer = None, None
    else:
        print(f"\n{'='*60}\nAblation — loading {model_label}\n{'='*60}")
        abl_model, abl_tokenizer = load_ablation_model(use_dpo)

    for temp in ABLATION_TEMPERATURES:
        print(f"\n--- {model_label} | temperature={temp} ---")
        res = generate_responses_llm(
            abl_model, abl_tokenizer, ablation_dataset, ablation_contexts,
            label=f"{model_label} (T={temp})",
            temperature_override=temp,
            compute_true_dahl=True,
            checkpoint_path=temp_ckpt_paths[temp],   # resume per-cell kalau disconnect
            save_every=3,
            checker_model=fixed_checker_model,             # Checker tetap, bukan abl_model
            checker_tokenizer=fixed_checker_tokenizer,
        )
        res = dahl_hallucination_audit(res)
        agg = aggregate_dahl_atomic(res, label=f"{model_label} (T={temp})")
        ablation_records.append({
            "model_label": model_label,
            "temperature": temp,
            **agg,
        })
        ablation_raw_results[f"{model_label}_T{temp}"] = res
        clean_mem()

    if abl_model is not None:
        del abl_model
        clean_mem()
    print(f"✅ {model_label} ablation selesai, model unloaded.")

if fixed_checker_model is not None:
    del fixed_checker_model
    clean_mem()
    print("✅ Fixed Checker model unloaded.")


ablation_df = pd.DataFrame(ablation_records)
ablation_csv_path = os.path.join(OUTPUT_DIR_EVAL, "ablation_dahl_score_validation.csv")
ablation_df.to_csv(ablation_csv_path, index=False)
print(f"\n✅ Ablation summary CSV: {ablation_csv_path}")
print(ablation_df.to_string(index=False))

ablation_json_path = os.path.join(OUTPUT_DIR_EVAL, "ablation_dahl_score_raw.json")
with open(ablation_json_path, "w", encoding="utf-8") as f:
    json.dump(ablation_raw_results, f, ensure_ascii=False, indent=2)
print(f"✅ Ablation raw JSON: {ablation_json_path}")

# ── Plot — replikasi gaya Figure 5 paper (DAHL Score & Hallucination Rate vs Temperature) ──
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for col, (lbl, color) in enumerate(zip(["Baseline", "DPO Model"], [C_BASE, C_DPO])):
    sub = ablation_df[ablation_df["model_label"] == lbl].sort_values("temperature")
    x_labels = sub["temperature"].astype(str)

    # Baris 1: DAHL Score
    ax_top = axes[0, col]
    bars = ax_top.bar(x_labels, sub["avg_dahl"].fillna(0), color=color, alpha=0.85)
    for bar, y in zip(bars, sub["avg_dahl"]):
        if y is not None:
            ax_top.text(bar.get_x() + bar.get_width() / 2, y + 0.01, f"{y:.3f}",
                        ha="center", va="bottom", fontsize=8)
    ax_top.set_title(f"{lbl} — DAHL Score vs Temperature")
    ax_top.set_ylabel("DAHL Score")
    ax_top.set_ylim(0, 1)
    ax_top.grid(axis="y", linestyle="--", alpha=0.5)

    # Baris 2: Hallucination Rate (DAHL)
    ax_bot = axes[1, col]
    if "hallucination_rate" in sub.columns:
        bars2 = ax_bot.bar(x_labels, sub["hallucination_rate"].fillna(0),
                            color=C_RED, alpha=0.85)
        for bar, y in zip(bars2, sub["hallucination_rate"]):
            if y is not None:
                ax_bot.text(bar.get_x() + bar.get_width() / 2, y + 0.01, f"{y:.1%}",
                            ha="center", va="bottom", fontsize=8)
    ax_bot.set_title(f"{lbl} — Hallucination Rate (DAHL) vs Temperature\n(threshold={DAHL_HALLUCINATION_THRESHOLD})")
    ax_bot.set_xlabel("Temperature")
    ax_bot.set_ylabel("Hallucination Rate")
    ax_bot.set_ylim(0, 1)
    ax_bot.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
ablation_plot_path = os.path.join(OUTPUT_DIR_EVAL, "ablation_dahl_score_validation.png")
plt.savefig(ablation_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Ablation plot disimpan: {ablation_plot_path}")

# ── Bandingkan Baseline vs DPO pada temperature default ───────────────
# untuk menjawab pertanyaan utama thesis: apakah DPO benar-benar
# menurunkan halusinasi menurut DAHL Score (bukan hanya proxy/TPS)?
default_temp_rows = ablation_df[ablation_df["temperature"] == TEMPERATURE]
if len(default_temp_rows) == 2:
    print(f"\n--- DAHL Score & Hallucination Rate @ temperature={TEMPERATURE} (default eval) ---")
    cols_to_show = [
        c for c in ["model_label", "avg_dahl", "hallucination_rate",
                    "hallucination_count", "n_valid", "n_excluded"]
        if c in default_temp_rows.columns
    ]
    print(default_temp_rows[cols_to_show].to_string(index=False))

print("\n✅ CELL 19 (validasi independen DAHL Score) selesai.")
# =========================================================
# CELL 20 — VALIDASI INDEPENDEN TAMBAHAN: NLI-Based
#            Hallucination Detection
# =========================================================
#
# TUJUAN: Menjawab kekhawatiran reviewer bahwa DAHL Score (Cell 9/19)
# masih berbagi formula yang sama dengan TPS training (cosine similarity
# + lexical overlap). Cell ini menambahkan metrik ketiga yang BENAR-BENAR
# berbeda total mekanismenya:
#
#   Metrik       | Model           | Mekanisme
#   -------------|-----------------|------------------------------------
#   TPS (Stage2) | MiniLM (embed)  | cosine sim + lexical overlap
#   DAHL (Cell19)| Qwen2 (LLM)     | atomic split + LLM binary judgment
#   NLI (Cell20) | DeBERTa (NLI)   | Natural Language Inference (entailment)
#
# CARA KERJA NLI:
#   cross-encoder/nli-deberta-v3-small mengklasifikasikan pasangan
#   (premise=context_dokumen, hypothesis=response) ke dalam 3 label:
#     - ENTAILMENT  : response didukung konteks  → bukan halusinasi
#     - NEUTRAL     : response tidak bertentangan → ambigu
#     - CONTRADICTION: response bertentangan      → halusinasi
#
#   Respons dianggap halusinasi jika label = CONTRADICTION
#   ATAU jika skor confidence ENTAILMENT sangat rendah (< NLI_ENTAIL_THRESHOLD)
#
# KEUNGGULAN vs TPS & DAHL:
#   1. Model benar-benar berbeda (DeBERTa, bukan MiniLM atau Qwen2)
#   2. Tidak ada cosine similarity → tidak ada tumpang tindih dengan TPS
#   3. Tidak butuh LLM besar → ringan di VRAM, bisa jalan paralel
#   4. Didukung literatur NLI untuk hallucination detection
#      (Honovich et al., 2022 — "TRUE: Re-evaluating factual consistency")
#
# OUTPUT:
#   - nli_validation_report.csv   : per-sample NLI label + skor
#   - nli_validation_summary.csv  : ringkasan Baseline vs DPO
#   - nli_validation.png          : visualisasi 3-panel
#
# CATATAN MEMORI:
#   DeBERTa-v3-small (~140MB) berjalan di CPU — tidak butuh GPU.
#   Bisa berjalan bersamaan dengan VRAM yang sudah bebas setelah
#   CELL 19 unload semua model.
# =========================================================

# ---- Konfigurasi NLI ----------------------------------------
NLI_MODEL_ID        = "cross-encoder/nli-deberta-v3-small"
NLI_ENTAIL_THRESHOLD = 0.50   # skor ENTAILMENT di bawah ini → ambigu/berisiko
NLI_MAX_CHARS       = 512     # potong teks untuk efisiensi (DeBERTa max 512 token)
NLI_BATCH_SIZE      = 8       # batch inference untuk kecepatan

# ---- Load results dari checkpoint jika kernel restart -------
# (Gunakan hasil evaluate utama dari Cell 13/14, bukan ablation Cell 19)
def _load_checkpoint(path: str, label: str) -> list:
    if path and os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"📂 {label}: {len(data)} sampel dimuat dari {path}")
        return data
    return []

if "baseline_results" not in globals() or not baseline_results:
    baseline_results = _load_checkpoint(
        os.path.join(OUTPUT_DIR_EVAL, "baseline_results_checkpoint.json"),
        "Baseline"
    )
if "dpo_results" not in globals() or not dpo_results:
    dpo_results = _load_checkpoint(
        os.path.join(OUTPUT_DIR_EVAL, "dpo_results_checkpoint.json"),
        "DPO Model"
    )
if not baseline_results or not dpo_results:
    raise RuntimeError(
        "❌ baseline_results / dpo_results kosong. "
        "Jalankan CELL 13–14 atau pastikan checkpoint JSON tersedia."
    )

# ---- Install & load NLI pipeline ----------------------------
print(f"\n{'='*60}")
print("CELL 20 — NLI-Based Independent Hallucination Validation")
print(f"{'='*60}")
print(f"NLI model  : {NLI_MODEL_ID}")
print(f"Mekanisme  : Natural Language Inference (DeBERTa cross-encoder)")
print(f"Device     : CPU (tidak butuh VRAM)")
print(f"Threshold  : ENTAILMENT score >= {NLI_ENTAIL_THRESHOLD} → bukan halusinasi")

try:
    from transformers import pipeline as hf_pipeline
    nli_pipe = hf_pipeline(
        "text-classification",
        model=NLI_MODEL_ID,
        device=-1,            # CPU — ringan, tidak konflik dengan VRAM
        truncation=True,
        max_length=512,
        top_k=None,           # return semua label sekaligus
    )
    print("✅ NLI pipeline siap.")
except Exception as e:
    raise RuntimeError(f"❌ Gagal load NLI model: {e}")


def _truncate_pair(context_text: str, response: str) -> tuple:
    """Potong premise+hypothesis agar tidak melebihi batas token DeBERTa."""
    half = NLI_MAX_CHARS // 2
    ctx_t  = context_text[:NLI_MAX_CHARS] if context_text else ""
    resp_t = response[:half] if response else ""
    return ctx_t, resp_t


def run_nli_on_results(results: list, label: str) -> list:
    """
    Jalankan NLI check untuk setiap sampel.

    Parameters
    ----------
    results : list[dict]
        Output dari generate_responses_llm() + hallucination_audit().
        Setiap dict harus punya key 'response', 'prompt', 'kode'.
        Konteks diambil dari all_contexts menggunakan index posisi.

    Returns
    -------
    list[dict] : hasil dengan tambahan field:
        nli_label         : ENTAILMENT / NEUTRAL / CONTRADICTION
        nli_entail_score  : skor confidence label ENTAILMENT [0,1]
        nli_contradict_score : skor label CONTRADICTION
        nli_hallucination : True jika CONTRADICTION atau entail < threshold
        nli_reason        : penjelasan flag
    """
    print(f"\n[NLI] Memproses {len(results)} sampel untuk {label}...")
    nli_results = []

    # Siapkan batch pasangan (premise, hypothesis)
    pairs         = []
    valid_indices = []

    for i, r in enumerate(results):
        response = r.get("response", "").strip()
        # Ambil konteks dari all_contexts (precomputed, sesuai urutan dataset)
        ctx_list = all_contexts[i] if i < len(all_contexts) else []
        ctx_text = " ".join(c.get("text", "") for c in ctx_list if c.get("text"))

        if not response or len(response) < 5 or not ctx_text.strip():
            # Tidak bisa dinilai → tandai sebagai ambigu
            nli_results.append({
                **r,
                "nli_label"           : "UNKNOWN",
                "nli_entail_score"    : 0.0,
                "nli_contradict_score": 0.0,
                "nli_hallucination"   : True,
                "nli_reason"          : "response atau konteks kosong — tidak dapat dinilai",
            })
            continue

        ctx_t, resp_t = _truncate_pair(ctx_text, response)
        pairs.append(f"{ctx_t} </s> {resp_t}")
        valid_indices.append(i)

    # Inference dalam batch
    all_preds = []
    for start in range(0, len(pairs), NLI_BATCH_SIZE):
        batch = pairs[start : start + NLI_BATCH_SIZE]
        try:
            preds = nli_pipe(batch)
            all_preds.extend(preds)
        except Exception as e:
            print(f"  [WARN] Batch {start//NLI_BATCH_SIZE} gagal: {e} — tandai UNKNOWN")
            all_preds.extend([None] * len(batch))

    # Gabungkan hasil ke nli_results
    pred_iter = iter(zip(valid_indices, all_preds))
    temp_nli  = {}
    for idx, preds in pred_iter:
        if preds is None:
            temp_nli[idx] = {
                "nli_label"           : "UNKNOWN",
                "nli_entail_score"    : 0.0,
                "nli_contradict_score": 0.0,
                "nli_hallucination"   : True,
                "nli_reason"          : "inference error",
            }
            continue

        # preds = list of {'label': ..., 'score': ...}
        score_map = {p["label"].upper(): p["score"] for p in preds}
        entail_s  = score_map.get("ENTAILMENT", 0.0)
        contra_s  = score_map.get("CONTRADICTION", 0.0)
        neutral_s = score_map.get("NEUTRAL", 0.0)

        # Label = argmax
        best_label = max(score_map, key=score_map.get)

        # Kriteria halusinasi:
        # 1. Label utama = CONTRADICTION
        # 2. ATAU skor ENTAILMENT < threshold (respons tidak cukup didukung)
        is_hallucination = (
            best_label == "CONTRADICTION"
            or entail_s < NLI_ENTAIL_THRESHOLD
        )
        reasons = []
        if best_label == "CONTRADICTION":
            reasons.append(
                f"NLI label=CONTRADICTION (score={contra_s:.3f})"
            )
        if entail_s < NLI_ENTAIL_THRESHOLD:
            reasons.append(
                f"ENTAILMENT score={entail_s:.3f} < threshold={NLI_ENTAIL_THRESHOLD}"
            )

        temp_nli[idx] = {
            "nli_label"           : best_label,
            "nli_entail_score"    : round(entail_s,  4),
            "nli_contradict_score": round(contra_s,  4),
            "nli_neutral_score"   : round(neutral_s, 4),
            "nli_hallucination"   : is_hallucination,
            "nli_reason"          : "; ".join(reasons) if reasons else "OK",
        }

    # Rakit hasil akhir (urutan asli)
    # Rebuild dari awal supaya urutan asli terjaga dan tidak ada duplikat.
    nli_results_final = []
    empty_iter = iter([r for r in nli_results if r.get("nli_label") == "UNKNOWN"
                       and "response" in r])
    for i, r in enumerate(results):
        if i in temp_nli:
            nli_results_final.append({**r, **temp_nli[i]})
        else:
            # Ambil entri yang sudah dibuat saat response/ctx kosong
            try:
                nli_results_final.append(next(empty_iter))
            except StopIteration:
                # Fallback: tandai UNKNOWN
                nli_results_final.append({
                    **r,
                    "nli_label": "UNKNOWN",
                    "nli_entail_score": 0.0,
                    "nli_contradict_score": 0.0,
                    "nli_hallucination": True,
                    "nli_reason": "assembly fallback — tidak dapat dinilai",
                })
    nli_results = nli_results_final

    print(f"  ✅ NLI selesai untuk {label}: {len(nli_results)} sampel.")
    return nli_results


def aggregate_nli(nli_results: list, label: str) -> dict:
    """Hitung hallucination rate berdasarkan NLI."""
    total     = len(nli_results)
    hall_cnt  = sum(1 for r in nli_results if r.get("nli_hallucination"))
    entail_scores = [
        r["nli_entail_score"] for r in nli_results
        if r.get("nli_label") not in ("UNKNOWN",)
    ]
    label_dist = {}
    for r in nli_results:
        lbl = r.get("nli_label", "UNKNOWN")
        label_dist[lbl] = label_dist.get(lbl, 0) + 1

    agg = {
        "model"                : label,
        "n_samples"            : total,
        "nli_hallucination_count": hall_cnt,
        "nli_hallucination_rate" : round(hall_cnt / total, 4) if total else 0.0,
        "avg_entail_score"     : round(float(np.mean(entail_scores)), 4) if entail_scores else 0.0,
        **{f"label_{k}": v for k, v in label_dist.items()},
    }
    print(
        f"\n[NLI] {label}: "
        f"halusinasi={hall_cnt}/{total} ({agg['nli_hallucination_rate']:.1%}) | "
        f"avg_entail={agg['avg_entail_score']:.4f} | "
        f"label_dist={label_dist}"
    )
    return agg


# ---- Jalankan NLI untuk Baseline dan DPO -------------------
baseline_nli = run_nli_on_results(baseline_results, "Baseline (pre-DPO)")
dpo_nli      = run_nli_on_results(dpo_results,      "DPO Model (post-DPO)")

baseline_nli_agg = aggregate_nli(baseline_nli, "Baseline (pre-DPO)")
dpo_nli_agg      = aggregate_nli(dpo_nli,      "DPO Model (post-DPO)")

# ---- Ringkasan perbandingan ---------------------------------
print(f"\n{'='*60}")
print("PERBANDINGAN NLI — Baseline vs DPO Model")
print(f"{'='*60}")
print(f"{'Metrik':<35} {'Baseline':>12} {'DPO':>12} {'Delta':>10}")
print("-" * 70)
for key in ["nli_hallucination_count", "nli_hallucination_rate", "avg_entail_score"]:
    b = baseline_nli_agg.get(key, 0)
    d = dpo_nli_agg.get(key, 0)
    delta = round(d - b, 4)
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "─")
    print(f"  {key:<33} {str(b):>12} {str(d):>12} {arrow}{abs(delta):>8.4f}")

# ---- Simpan CSV per-sampel ----------------------------------
def flatten_nli(nli_results, model_label):
    rows = []
    for r in nli_results:
        rows.append({
            "kode"                  : r.get("kode", ""),
            "topik"                 : r.get("topik", ""),
            "model_label"           : model_label,
            "prompt"                : r.get("prompt", ""),
            "response"              : r.get("response", ""),
            "nli_label"             : r.get("nli_label", ""),
            "nli_entail_score"      : r.get("nli_entail_score", ""),
            "nli_contradict_score"  : r.get("nli_contradict_score", ""),
            "nli_neutral_score"     : r.get("nli_neutral_score", ""),
            "nli_hallucination"     : r.get("nli_hallucination", ""),
            "nli_reason"            : r.get("nli_reason", ""),
            # Sertakan TPS/DAHL lama untuk perbandingan lintas metrik
            "tps_dahl_score"        : r.get("dahl_score", ""),
            "tps_hallucination_flag": r.get("hallucination_flag", ""),
        })
    return rows

nli_detail_df = pd.DataFrame(
    flatten_nli(baseline_nli, "Baseline (pre-DPO)")
    + flatten_nli(dpo_nli,    "DPO Model (post-DPO)")
)
nli_csv_path = os.path.join(OUTPUT_DIR_EVAL, "nli_validation_report.csv")
nli_detail_df.to_csv(nli_csv_path, index=False, encoding="utf-8-sig")
print(f"\n✅ NLI per-sample CSV   : {nli_csv_path}")

nli_summary_df = pd.DataFrame([baseline_nli_agg, dpo_nli_agg])
nli_summary_csv = os.path.join(OUTPUT_DIR_EVAL, "nli_validation_summary.csv")
nli_summary_df.to_csv(nli_summary_csv, index=False, encoding="utf-8-sig")
print(f"✅ NLI summary CSV      : {nli_summary_csv}")

# ---- Visualisasi 3-panel NLI --------------------------------
fig_nli, axes_nli = plt.subplots(1, 3, figsize=(15, 5))
fig_nli.suptitle(
    "NLI-Based Independent Hallucination Validation\n"
    f"Model: {NLI_MODEL_ID} | Entailment threshold={NLI_ENTAIL_THRESHOLD}",
    fontsize=12, fontweight="bold"
)

# Panel 1: Hallucination Rate (NLI)
ax_hall = axes_nli[0]
h_rates = [
    baseline_nli_agg["nli_hallucination_rate"],
    dpo_nli_agg["nli_hallucination_rate"],
]
bars = ax_hall.bar(["Baseline", "DPO Model"], h_rates, color=[C_BASE, C_DPO], alpha=0.85, width=0.4)
for bar, rate in zip(bars, h_rates):
    ax_hall.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f"{rate:.1%}", ha="center", va="bottom", fontweight="bold", fontsize=11
    )
ax_hall.set_title("Hallucination Rate (NLI)")
ax_hall.set_ylabel("Rate"); ax_hall.set_ylim(0, 1)
ax_hall.grid(axis="y", linestyle="--", alpha=0.6)
ax_hall.axhline(
    y=baseline_nli_agg["nli_hallucination_rate"],
    color=C_RED, linestyle=":", linewidth=1.2, label="Baseline rate"
)
ax_hall.legend(fontsize=8)

# Panel 2: Avg Entailment Score
ax_entail = axes_nli[1]
e_scores = [
    baseline_nli_agg["avg_entail_score"],
    dpo_nli_agg["avg_entail_score"],
]
bars2 = ax_entail.bar(["Baseline", "DPO Model"], e_scores, color=[C_BASE, C_DPO], alpha=0.85, width=0.4)
for bar, score in zip(bars2, e_scores):
    ax_entail.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
        f"{score:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=10
    )
ax_entail.axhline(
    y=NLI_ENTAIL_THRESHOLD, color=C_RED, linestyle=":",
    linewidth=1.2, label=f"Threshold ({NLI_ENTAIL_THRESHOLD})"
)
ax_entail.set_title("Avg Entailment Score (NLI)")
ax_entail.set_ylabel("Score"); ax_entail.set_ylim(0, 1)
ax_entail.legend(fontsize=8)
ax_entail.grid(axis="y", linestyle="--", alpha=0.6)

# Panel 3: NLI Label Distribution per model (stacked bar)
ax_dist = axes_nli[2]
all_labels = ["ENTAILMENT", "NEUTRAL", "CONTRADICTION", "UNKNOWN"]
colors_label = {"ENTAILMENT": C_GRN, "NEUTRAL": "#AAAAAA",
                "CONTRADICTION": C_RED, "UNKNOWN": "#DDDDDD"}
x_pos  = np.arange(2)
bottoms = [0, 0]
for lbl in all_labels:
    key  = f"label_{lbl}"
    vals = [
        baseline_nli_agg.get(key, 0) / max(baseline_nli_agg["n_samples"], 1),
        dpo_nli_agg.get(key, 0)      / max(dpo_nli_agg["n_samples"], 1),
    ]
    ax_dist.bar(
        x_pos, vals, bottom=bottoms,
        color=colors_label[lbl], alpha=0.85, label=lbl, width=0.4
    )
    bottoms = [b + v for b, v in zip(bottoms, vals)]
ax_dist.set_xticks(x_pos)
ax_dist.set_xticklabels(["Baseline", "DPO Model"])
ax_dist.set_title("NLI Label Distribution")
ax_dist.set_ylabel("Proportion")
ax_dist.set_ylim(0, 1.05)
ax_dist.legend(fontsize=8, loc="upper right")
ax_dist.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
nli_plot_path = os.path.join(OUTPUT_DIR_EVAL, "nli_validation.png")
plt.savefig(nli_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ NLI plot disimpan    : {nli_plot_path}")

# ---- Tabel perbandingan 3 metrik: TPS, DAHL, NLI -----------
# Ini adalah tabel utama untuk paper — menunjukkan konsistensi
# reduksi halusinasi lintas TIGA metode pengukuran yang berbeda.
print(f"\n{'='*65}")
print("TABEL KONSISTENSI — Hallucination Rate: TPS vs DAHL vs NLI")
print(f"{'='*65}")
print(f"{'Metrik':<25} {'Baseline':>12} {'DPO':>12} {'Reduksi':>10}")
print("-" * 65)

# TPS (dari evaluasi utama Cell 13/14)
tps_base = baseline_agg.get("hallucination_rate", 0)
tps_dpo  = dpo_agg.get("hallucination_rate", 0)
print(
    f"  {'TPS Proxy (Cell 13/14)':<23} {tps_base:>11.1%} {tps_dpo:>11.1%} "
    f"{(tps_base - tps_dpo):>9.1%}"
)

# DAHL Atomic (dari Cell 19)
if "ablation_df" in dir() or "ablation_df" in globals():
    try:
        dahl_base_row = ablation_df[
            (ablation_df["model_label"] == "Baseline") &
            (ablation_df["temperature"] == TEMPERATURE)
        ]
        dahl_dpo_row  = ablation_df[
            (ablation_df["model_label"] == "DPO Model") &
            (ablation_df["temperature"] == TEMPERATURE)
        ]
        if len(dahl_base_row) and len(dahl_dpo_row):
            dahl_base = dahl_base_row["hallucination_rate"].values[0]
            dahl_dpo  = dahl_dpo_row["hallucination_rate"].values[0]
            print(
                f"  {'DAHL Atomic (Cell 19)':<23} {dahl_base:>11.1%} {dahl_dpo:>11.1%} "
                f"{(dahl_base - dahl_dpo):>9.1%}"
            )
    except Exception:
        print(f"  {'DAHL Atomic (Cell 19)':<23} {'(jalankan Cell 19)':>35}")
else:
    print(f"  {'DAHL Atomic (Cell 19)':<23} {'(jalankan Cell 19 dulu)':>35}")

# NLI (Cell 20 — baru saja dihitung)
nli_base = baseline_nli_agg["nli_hallucination_rate"]
nli_dpo  = dpo_nli_agg["nli_hallucination_rate"]
print(
    f"  {'NLI DeBERTa (Cell 20)':<23} {nli_base:>11.1%} {nli_dpo:>11.1%} "
    f"{(nli_base - nli_dpo):>9.1%}"
)
print(f"{'='*65}")
print(
    "Catatan: Ketiga metrik menggunakan model/mekanisme berbeda.\n"
    "Konsistensi reduksi halusinasi lintas metrik menguatkan klaim\n"
    "bahwa DPO alignment benar-benar mengurangi halusinasi.\n"
    "TPS=cosine+lexical, DAHL=atomic LLM judgment, NLI=DeBERTa entailment."
)

print("\n✅ CELL 20 (NLI-based independent validation) selesai.")
print(f"   Output:")
print(f"     • nli_validation_report.csv   — detail per sampel")
print(f"     • nli_validation_summary.csv  — ringkasan Baseline vs DPO")
print(f"     • nli_validation.png          — 3-panel visualisasi")

In [ ]:
# CELL 21 — STATISTICAL SIGNIFICANCE TESTING
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
import pandas as pd, numpy as np, os

df = pd.read_csv(os.path.join(OUTPUT_DIR_EVAL, "evaluation_per_sample.csv"))

# Split by model_label — the actual column name from flatten_results()
base_df = df[df["model_label"] == "Baseline (pre-DPO)"].reset_index(drop=True)
dpo_df  = df[df["model_label"] == "DPO Model (post-DPO)"].reset_index(drop=True)

# Align by kode to ensure paired order is identical
base_df = base_df.sort_values("kode").reset_index(drop=True)
dpo_df  = dpo_df.sort_values("kode").reset_index(drop=True)

print("=" * 60)
print("WILCOXON SIGNED-RANK TEST — Continuous Metrics (n=200 paired)")
print("=" * 60)
# Columns from flatten_results(): dahl_score, dim1_relevance, dim2_grounding, dim3_lexical
for col in ["dahl_score", "dim1_relevance", "dim2_grounding", "dim3_lexical"]:
    stat, p = wilcoxon(base_df[col], dpo_df[col], alternative="less")
    print(f"  {col:<22} W={stat:.1f}  p={p:.4f}  {'✓ sig (p<0.05)' if p < 0.05 else '✗ not sig'}")

print("\n" + "=" * 60)
print("McNEMAR TEST — Binary Hallucination Labels (n=200 paired)")
print("=" * 60)
# Column from flatten_results(): hallucination_flag
b = base_df["hallucination_flag"].astype(int).values
d = dpo_df["hallucination_flag"].astype(int).values

n11 = ((b==1) & (d==1)).sum()  # both hallucinated
n10 = ((b==1) & (d==0)).sum()  # baseline hallucinated, DPO didn't  ← key cell
n01 = ((b==0) & (d==1)).sum()  # DPO hallucinated, baseline didn't  ← key cell
n00 = ((b==0) & (d==0)).sum()  # both OK

from statsmodels.stats.contingency_tables import mcnemar
result = mcnemar([[n11, n10], [n01, n00]], exact=True)
print(f"  TPS hallucination_flag")
print(f"    n10 (base=hall, dpo=ok) : {n10}")
print(f"    n01 (base=ok,  dpo=hall): {n01}")
print(f"    p-value (McNemar exact) : {result.pvalue:.4f}  "
      f"{'✓ sig' if result.pvalue < 0.05 else '✗ not sig'}")